# 02 · 特征工程

**原则**：
- 前三轮只用逐行变换和 train + test 的**无标签**统计（计数、比例、去重计数），不使用 TotalKg。
- 第四轮尝试基于成绩的特征，一律用嵌套折外计算，防止泄漏。

| 轮次 | 特征块 | 结果 |
|---|---|---|
| 1（实验 07） | `P` 记录精度、`T` 跨地参赛、`M` 比赛背景 | P、M 有效，合计 −0.67 kg |
| 2（实验 09） | `M2` 比赛背景加强、`D` 组别背景、`H` 运动员历史、`S` DOTS 系数、`C` 比赛 / 运动员 ID | 只有 H 达标（−0.13） |
| 3（实验 10） | `H2` 运动员生涯、`E` 比赛经验结构、`PM` 按模型挑特征 | 三块都达标，合计 −0.20 |
| 4（实验 14） | `AS` 运动员实力、`MS` 比赛实力（基于成绩，嵌套折外） | 都变差，不用 |

**消融协议**：
- 固定五折，外层训练部分内留 10% 早停，直接用早停模型给外层验证打分。
- 每个变体训练 LightGBM、XGBoost、CatBoost 三个模型，再做三模型 NNLS 融合。
- 只比较同一折上相对基线的 RMSE 差：融合的平均 ΔRMSE ≤ −0.05 kg 且至少 4/5 折更好才保留。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 读取与核对原始数据、探索分析、记录精度与「双胞胎」、数据质量检查、固定五折 |
| **02_feature_engineering** | 四轮特征（记录精度、跨地参赛、比赛背景、运动员历史与生涯、基于成绩的特征）及五折消融（实验 07、09、10、14） |
| **03_model_selection** | 早期基线、CatBoost 参数、其他模型与相对力量目标、堆叠方式、目标变量工程、模型大比较（实验 08、13） |
| **04_final_model** | 最终集成：8 个模型 × 5 折 × 3 种子 + 全量重训、NNLS 融合、双胞胎 + 同一运动员平滑、提交文件（实验 11、12、15） |
| **05_error_analysis** | 最终残差、同一运动员与双胞胎结构、第二层做法比较、数据清洗实验、未采用的方法、交叉验证与公开榜 |

原始数据在 `data/`；结果保存在 `artifacts/` 下与 notebook 同名的目录。「实验 07–15」是实验的原始编号，代码注释和 RESULTS.md 沿用这些编号。

**训练开关**：默认全部为 `False`，只读取 `artifacts/` 中已保存的结果并生成表格和图，Colab 普通 CPU 运行时即可，几分钟跑完。

把某一项改为 `True` 并选择 **T4 GPU** 运行时，会按原协议重新训练对应的实验。每个模型训练完立即写入 `artifacts/`，断线后从头运行会跳过已完成的部分。**一次只打开一个开关**（不同实验的代码共用变量名），跑完后重启运行时再打开下一个。

In [ ]:
TRAIN = {
    'round1': False,   # 实验 07：第一轮特征 6 个变体 × 3 个模型 × 5 折 + CatBoost 参数 + 双胞胎（GPU，约 1.5–2 小时）
    'round2': False,   # 实验 09：第二轮特征（GPU，约 1.5–2 小时）
    'round3': False,   # 实验 10：第三轮特征 + 第二层平滑比较（GPU，约 1–1.5 小时）
    'round4': False,   # 实验 14：基于成绩的特征 + id 检查 + 目标变量工程（GPU，约 1–1.5 小时）
}

In [ ]:
# 环境、路径与通用设置（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import nnls
from sklearn.model_selection import KFold, train_test_split

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')


def find_project_dir():
    candidates = []
    if os.environ.get('PL_PROJECT_DIR'):
        candidates.append(Path(os.environ['PL_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 PL_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                 # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'           # 每本 notebook 的结果在与它同名的子目录
SEED = 42                                       # 固定五折：KFold(5, shuffle=True, random_state=42)
N_FOLDS = 5
INNER_STOP_FRACTION = 0.10                      # 外层训练部分内留 10% 早停
MAX_ROUNDS = 20000
MODELS = ['lgb', 'xgb', 'catboost']             # 特征消融用的三个模型（每折内的运行顺序：快的先跑）
RUN_ONLY = None                                 # 只跑部分变体时写成列表；None 表示全部
KEEP_DELTA = -0.05                              # kg：平均 ΔRMSE ≤ −0.05 且 ≥ 4/5 折更好才保留
EXPECTED_SHA256 = {'train.csv': '62319814116506fb95a51f1185dbacdb170ab055642ee4c347a30584093d0856',
                   'test.csv': '563c6fa29f9e535740ef1f09b25954f00fbe242c81c0e5a7343dc18aa962ba46'}
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
VERSIONS = {'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

TRAIN = globals().get('TRAIN', {})              # 01 与 05 没有训练开关
if sum(bool(v) for v in TRAIN.values()) > 1:
    raise ValueError('一次只打开一个训练开关：不同实验的代码共用变量名。跑完一个、重启运行时后再打开下一个')
TRAINING = any(TRAIN.values())


def ensure_training_env():
    """只在训练时调用：安装固定版本的 CatBoost 与 pytabkit，导入训练用的库，确认 Colab 分配了 GPU。"""
    global catboost, lgb, xgb, torch, RealMLP_TD_Regressor, TabM_D_Regressor, PYTABKIT_ERROR, GPU_NAME, VERSIONS
    to_install = []
    try:
        installed_catboost = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed_catboost = None
    if installed_catboost != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启运行时后从头运行')
        to_install.append(f'catboost=={CATBOOST_VERSION}')
    try:
        importlib.metadata.version('pytabkit')
    except importlib.metadata.PackageNotFoundError:
        to_install.append('pytabkit')
    if to_install:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
        importlib.invalidate_caches()
    import catboost
    import lightgbm as lgb
    import torch
    import xgboost as xgb
    try:
        from pytabkit import RealMLP_TD_Regressor, TabM_D_Regressor
        PYTABKIT_ERROR = None
    except Exception as exc:                    # 安装或导入失败时其余模型照常运行
        RealMLP_TD_Regressor = TabM_D_Regressor = None
        PYTABKIT_ERROR = repr(exc)
    warnings.filterwarnings('ignore', category=UserWarning)
    if catboost.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')
    if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
        raise RuntimeError('训练需要 GPU：请在 Colab「更改运行时类型」中选择 T4 GPU 后从头运行')
    GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                              text=True, capture_output=True, check=False).stdout.strip().splitlines()[0]
    VERSIONS = {'catboost': catboost.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
                'torch': torch.__version__,
                'pytabkit': None if PYTABKIT_ERROR else importlib.metadata.version('pytabkit'),
                'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
    print('GPU         :', GPU_NAME)
    print('版本         :', VERSIONS)
    if PYTABKIT_ERROR:
        print('pytabkit 不可用，RealMLP / TabM 会在预检中跳过：', PYTABKIT_ERROR)


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')
if TRAINING:
    print('打开的训练开关：', [k for k, v in TRAIN.items() if v])

## 0. 读取数据与固定五折

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rmse(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    return float(np.sqrt(np.mean((a - b) ** 2)))


for name, expected in EXPECTED_SHA256.items():
    require(sha256_file(DATA_DIR / name) == expected, f'data/{name} 与原始数据不同；请检查数据文件')

TARGET = 'TotalKg'
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
RAW_FEATURES = [c for c in test.columns if c != 'id']
CAT = [c for c in RAW_FEATURES if c not in RAW_NUM]          # 不依赖 object / str dtype（pandas 2 与 3 都可用）
require(train.shape == (105000, 18) and test.shape == (45000, 17), '数据形状不符')
require(len(CAT) == 13 and all(pd.api.types.is_numeric_dtype(train[c]) for c in RAW_NUM)
        and not any(pd.api.types.is_numeric_dtype(train[c]) for c in CAT), '列分组不符')
require(not train[RAW_FEATURES].isna().any().any() and not test[RAW_FEATURES].isna().any().any(), '出现空值')
y = train[TARGET].to_numpy(dtype='float64')
n_train = len(train)

FOLD_IDS = np.zeros(n_train, dtype=int)
for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.arange(n_train)), start=1):
    FOLD_IDS[valid] = k
print(f'train {train.shape}  test {test.shape}；每折验证行数：{np.bincount(FOLD_IDS)[1:].tolist()}')

## 1. 第一轮特征（实验 07）

**为什么做**：v2 融合（OOF RMSE 77.302）的残差在下面这些分组上明显不为 0，说明模型没有学到这些信号。

| 信号 | 残差均值（kg） | 解释 |
|---|---|---|
| 年龄整数 / x.5 | +2.6 / −3.6；USAPL 内 **+19.0 / −5.5** | x.5 表示只知道出生年份 |
| 体重记录精度 | 整数 −2.9，两位小数 +1.3 | 称重与录入精度对应赛事级别 |
| 跨州参赛 / 同州 | +3.6 / −1.8 | 去外州比赛的多是更高级别的赛事 |
| 出国参赛 / 本国 | +1.6 / −0.5 | 国际赛事 |
| 比赛规模最小的 1/8 | −3.6 | 小型地方赛 |

「比赛」近似为 联盟 × 年份 × 比赛国 × 比赛州。

| 块 | 新列 |
|---|---|
| `F0` 基线 | v2 的 19 列：3 数值 + 13 类别 + 体重级别 3 列（`WeightClass_num`、`Is_SHW`、`BW_minus_Class`） |
| `P` 记录精度 | `age_half`（1 = x.5）、`bw_precision`（0 整数 / 1 x.5 / 2 其他一位小数 / 3 两位小数）、`bw_lb_int`（体重恰为整磅） |
| `T` 跨地参赛 | `abroad`、`other_state`（居住地未知时为 NaN） |
| `M` 比赛背景 | 比赛人数、同性别同装备人数、联盟当年人数；比赛内跨州、出国、x.5 年龄、装备、男性的比例；比赛平均年龄 |

In [ ]:
both = pd.concat([train[RAW_FEATURES], test[RAW_FEATURES]], ignore_index=True)

# F0：v2 的体重级别特征
weight_class = pd.to_numeric(both['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
F0_EXTRA = pd.DataFrame({
    'WeightClass_num': weight_class,
    'Is_SHW': both['WeightClassKg'].str.endswith('+').astype('float64'),
    'BW_minus_Class': both['BodyweightKg'] - weight_class,
})

# P：记录精度（树按阈值切数值，学不到「是不是 .5」「是不是整数」）
age2 = (both['Age'] * 2).round()
require(np.allclose(both['Age'] * 2, age2), 'Age 不是 0.5 的整数倍')
bw = both['BodyweightKg']
is_int = np.isclose(bw, bw.round())
is_1dec = np.isclose(bw * 10, (bw * 10).round())
is_half = is_1dec & ~is_int & np.isclose((bw * 10).round() % 10, 5)
pounds = bw / 0.45359237
P = pd.DataFrame({
    'age_half': (age2 % 2).astype('float64'),
    'bw_precision': np.select([is_int, is_half, is_1dec], [0.0, 1.0, 2.0], default=3.0),
    'bw_lb_int': (np.abs(pounds - pounds.round()) < 0.015).astype('float64'),
})

# T：跨地参赛（居住地未知时为 NaN）
abroad = (both['Country'] != both['MeetCountry']).astype('float64').where(both['Country'] != 'Unknown')
state_known = ((both['State'] != 'Unknown') & (both['MeetState'] != 'Unknown')
               & (both['Country'] == both['MeetCountry']))
other_state = (both['State'] != both['MeetState']).astype('float64').where(state_known)
T = pd.DataFrame({'abroad': abroad, 'other_state': other_state})

# M：比赛背景；「比赛」≈ 联盟 × 年份 × 比赛国 × 比赛州（train+test 无标签统计）
meet = (both['Federation'] + '|' + both['MeetYear'].astype(str) + '|'
        + both['MeetCountry'] + '|' + both['MeetState'])


def by_meet(values):
    return values.groupby(meet).transform('mean')


M = pd.DataFrame({
    'meet_n': meet.map(meet.value_counts()).astype('float64'),
    'meet_sexeq_n': both.groupby([meet, both['Sex'], both['Equipment']])['Age'].transform('size').astype('float64'),
    'fed_year_n': both.groupby(['Federation', 'MeetYear'])['Age'].transform('size').astype('float64'),
    'meet_share_other_state': by_meet(other_state),
    'meet_share_abroad': by_meet(abroad),
    'meet_share_age_half': by_meet(P['age_half']),
    'meet_share_equipped': by_meet((both['Equipment'] != 'Raw').astype('float64')),
    'meet_share_male': by_meet((both['Sex'] == 'M').astype('float64')),
    'meet_mean_age': by_meet(both['Age']),
})

BLOCKS = {'P': P, 'T': T, 'M': M}
VARIANTS = {'F0': [], 'P': ['P'], 'T': ['T'], 'M': ['M'], 'PT': ['P', 'T'], 'ALL': ['P', 'T', 'M']}
for name, frame in {'F0': F0_EXTRA, **BLOCKS}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')

_FRAMES = {}


def frames_for(variant, model):
    """(train, test) 特征表。CatBoost 的类别为字符串；LGB / XGB 为 category（词表来自 train+test，无标签）。"""
    kind = 'str' if model == 'catboost' else 'category'
    if (variant, kind) not in _FRAMES:
        frame = pd.concat([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in VARIANTS[variant]] + [both[CAT]], axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        frame[CAT] = frame[CAT].astype(str).astype(object) if kind == 'str' else frame[CAT].astype('category')
        _FRAMES[(variant, kind)] = (frame.iloc[:n_train].reset_index(drop=True),
                                    frame.iloc[n_train:].reset_index(drop=True))
    return _FRAMES[(variant, kind)]


print({v: frames_for(v, 'lgb')[0].shape[1] for v in VARIANTS})
summary_new = pd.concat([P, T, M], axis=1)
display(pd.DataFrame({'train_mean': summary_new.iloc[:n_train].mean(), 'test_mean': summary_new.iloc[n_train:].mean(),
                      'nan_share': summary_new.isna().mean()}).round(4))

## 2. 第二轮特征（实验 09）

基线 `R1` = F0 + P + T + M（33 列）。

| 块 | 内容 |
|---|---|
| `M2` 比赛背景加强 | 比赛内的州数、国家数、级别数；药检、无装备、多层装备、两位小数体重、青年、大师的比例；年龄标准差；同性别平均体重；全国赛规模 |
| `D` 组别背景 | 组别（比赛 × 性别 × 装备 × 级别）人数，组内体重和年龄排名；完全重复行与核心列重复行的个数 |
| `H` 运动员历史 | 近似同一运动员（性别 + 国籍 + 州 + 联盟 + 出生年）的出现次数、出现年数、距第一次 / 最后一次的年数、用过几种装备、体重偏差 |
| `S` DOTS 系数 | 力量举通用的体重-性别系数（TotalKg = DOTS 分 × 系数） |
| `C` ID（只给 CatBoost） | 比赛 ID、运动员 ID，用 CatBoost 的有序目标统计编码 |

出生年 = `floor(MeetYear − Age)`：x.5 年龄时恰好是出生年，整数年龄可能差 1。

In [ ]:
_FRAMES.clear()                     # 07 的变体表不再需要，释放内存
CORE9 = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']
sex, equipment = both['Sex'], both['Equipment']
state = both['State'].where(both['State'] != 'Unknown')
country = both['Country'].where(both['Country'] != 'Unknown')
national = both['Federation'] + '|' + both['MeetYear'].astype(str) + '|' + both['MeetCountry']


def counts(key):
    return key.map(key.value_counts()).astype('float64')


def n_distinct(values, key):
    return values.groupby(key).transform('nunique').astype('float64')


# M2：比赛背景加强（meet 与 by_meet 来自 07 的特征单元）
M2 = pd.DataFrame({
    'meet_n_states': n_distinct(state, meet),
    'meet_n_countries': n_distinct(country, meet),
    'meet_n_wclass': n_distinct(both['WeightClassKg'], meet),
    'meet_share_tested': by_meet((both['Tested'] == 'Yes').astype('float64')),
    'meet_share_raw': by_meet((equipment == 'Raw').astype('float64')),
    'meet_share_multi': by_meet((equipment == 'Multi-ply').astype('float64')),
    'meet_share_bw2dec': by_meet((P['bw_precision'] == 3).astype('float64')),
    'meet_share_junior': by_meet((both['Age'] < 24).astype('float64')),
    'meet_share_master': by_meet((both['Age'] >= 40).astype('float64')),
    'meet_age_std': both['Age'].groupby(meet).transform('std'),
    'meet_sex_bw_mean': both['BodyweightKg'].groupby([meet, sex]).transform('mean'),
    'nat_n': counts(national),
    'nat_n_meetstates': n_distinct(both['MeetState'], national),
    'nat_n_states': n_distinct(state, national),
})

# D：组别背景
division = meet + '|' + sex + '|' + equipment + '|' + both['WeightClassKg']
age_division = meet + '|' + sex + '|' + equipment + '|' + both['AgeClass']
exact_key = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy())
core_key = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy())
D = pd.DataFrame({
    'div_n': counts(division),
    'div_bw_rank': both['BodyweightKg'].groupby(division).rank(pct=True),
    'div_age_rank': both['Age'].groupby(division).rank(pct=True),
    'agediv_n': counts(age_division),
    'dup_exact_n': counts(exact_key) - 1,
    'dup_core_n': counts(core_key) - 1,
})

# H：近似同一运动员的历史（出生年 = floor(MeetYear − Age)：x.5 年龄时恰为出生年，整数年龄可能差 1）
valid = (both['Country'] != 'Unknown').to_numpy()
birth = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).astype(str)
athlete = sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation'] + '|' + birth
athlete0 = sex + '|' + both['Country'] + '|' + both['State'] + '|' + birth
year = both['MeetYear'].astype('float64')
H = pd.DataFrame({
    'ath_n': counts(athlete),
    'ath_n_years': n_distinct(year, athlete),
    'ath_years_since_first': year - year.groupby(athlete).transform('min'),
    'ath_years_to_last': year.groupby(athlete).transform('max') - year,
    'ath_n_equipment': n_distinct(equipment, athlete),
    'ath_bw_delta': both['BodyweightKg'] - both['BodyweightKg'].groupby(athlete).transform('mean'),
    'ath0_n': counts(athlete0),
    'ath0_n_feds': n_distinct(both['Federation'], athlete0),
})
H.loc[~valid] = np.nan


# S：DOTS 系数（TotalKg = DOTS 分 × dots_scale）
def dots_scale(sex_values, bw_values):
    s = np.asarray(sex_values).astype(str)
    bw = np.asarray(bw_values, dtype='float64')
    m, f = np.clip(bw, 40, 210), np.clip(bw, 40, 150)
    poly_m = -1.0930e-6 * m**4 + 7.391293e-4 * m**3 - 0.1918759221 * m**2 + 24.0900756 * m - 307.75076
    poly_f = -1.0706e-6 * f**4 + 5.158568e-4 * f**3 - 0.1126655495 * f**2 + 13.6175032 * f - 57.96288
    return np.where(s == 'M', poly_m, np.where(s == 'F', poly_f, 0.5 * (poly_m + poly_f))) / 500.0


S = pd.DataFrame({'dots_scale': dots_scale(sex, both['BodyweightKg'])})

# C：只给 CatBoost 的高基数 ID（有序目标统计编码）
C = pd.DataFrame({'meet_id': meet, 'athlete_id': (athlete + '|' + equipment).where(valid, 'unknown')})

BLOCKS2 = {'M2': M2, 'D': D, 'H': H, 'S': S}
CB_ONLY = {'C': C}
for name, frame in {**BLOCKS2, **CB_ONLY}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
for name, frame in BLOCKS2.items():
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
require(S['dots_scale'].min() > 0.3, 'DOTS 系数异常')

R1_BLOCKS = ['P', 'T', 'M']
VARIANT_BLOCKS = {'R1': [], 'M2': ['M2'], 'D': ['D'], 'H': ['H'], 'S': ['S'], 'C': ['C']}
MODELS_FOR = {'C': ['catboost']}      # C 只改变 CatBoost；融合时 LightGBM / XGBoost 用 R1
SCREEN = [v for v in VARIANT_BLOCKS if RUN_ONLY is None or v in RUN_ONLY or v == 'R1']
_FRAMES2 = {}


def models_for(variant):
    return MODELS_FOR.get(variant, MODELS)


def frames2(variant, model):
    """(train, test, 类别列)。R1 = F0 + P + T + M；C 块只加给 CatBoost。"""
    blocks = VARIANT_BLOCKS[variant]
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES2:
        nums = ([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in R1_BLOCKS]
                + [BLOCKS2[b] for b in blocks if b in BLOCKS2])
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES2[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES2[key]


new_cols = pd.concat([M2, D, H, S], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print('C 块类别数：', {c: int(C[c].nunique()) for c in C.columns})
print('各变体列数（LightGBM / XGBoost）：', {v: frames2(v, 'lgb')[0].shape[1] for v in VARIANT_BLOCKS if v != 'C'},
      '｜CatBoost C：', frames2('C', 'catboost')[0].shape[1])

## 3. 第三轮特征（实验 10）

基线 `R2` = 第二轮的 ALL2（R1 + H，41 列）。

| 块 | 内容 | 为什么 |
|---|---|---|
| `H2` 运动员生涯 | 参加过的最大比赛规模、最多来自几个州的比赛、是否出国比赛过、换过几个级别、体重变化范围、第一次出现的年龄、同一年出现次数、药检赛比例、出生年 ±1 的模糊匹配计数 | `H` 有效说明「这个人是谁」重要；去过多高级别的比赛把 M 和 H 结合起来 |
| `E` 比赛经验结构 | 选手平均出现次数、平均经验年数、老将（经验 ≥ 3 年）比例、新人比例 | 经验丰富的选手多的比赛级别更高 |
| `PM` 按模型挑特征 | CatBoost 加 `M2` + `C`；XGBoost 加 `M2` + `S` + `D`；LightGBM 不加 | 第二轮中各模型单独有效的块不同，也增加融合多样性 |

In [ ]:
_FRAMES2.clear()                    # 09 的变体表不再需要，释放内存
birth_num = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int)


def fuzzy_count(prefix_key):
    """同前缀、出生年为 b−1、b、b+1 的行数（整数年龄推算的出生年可能差 1）。"""
    exact = prefix_key + '|' + birth_num.astype(str)
    table = exact.value_counts()
    total = exact.map(table).astype('float64')
    for d in (-1, 1):
        total = total + (prefix_key + '|' + (birth_num + d).astype(str)).map(table).fillna(0)
    return total


# H2：运动员生涯（athlete、valid 来自 09 的特征单元；M、T 来自 07；M2 来自 09）
bw = both['BodyweightKg']
H2 = pd.DataFrame({
    'ath_max_meet_states': M2['meet_n_states'].groupby(athlete).transform('max'),
    'ath_mean_meet_states': M2['meet_n_states'].groupby(athlete).transform('mean'),
    'ath_max_meet_n': M['meet_n'].groupby(athlete).transform('max'),
    'ath_max_nat_states': M2['nat_n_states'].groupby(athlete).transform('max'),
    'ath_ever_abroad': T['abroad'].groupby(athlete).transform('max'),
    'ath_n_wclass': n_distinct(both['WeightClassKg'], athlete),
    'ath_bw_range': bw.groupby(athlete).transform('max') - bw.groupby(athlete).transform('min'),
    'ath_first_age': both['Age'].groupby(athlete).transform('min'),
    'ath_n_same_year': both.groupby([athlete, both['MeetYear']])['Age'].transform('size').astype('float64'),
    'ath_share_tested': (both['Tested'] == 'Yes').astype('float64').groupby(athlete).transform('mean'),
    'ath_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation']),
    'ath0_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State']),
})
H2.loc[~valid] = np.nan

# E：比赛的经验结构
experience = H['ath_years_since_first']
E = pd.DataFrame({
    'meet_mean_ath_n': H['ath_n'].groupby(meet).transform('mean'),
    'meet_mean_ath_exp': experience.groupby(meet).transform('mean'),
    'meet_share_veterans': (experience >= 3).astype('float64').where(experience.notna()).groupby(meet).transform('mean'),
    'meet_share_newcomers': (H['ath_n'] == 1).astype('float64').where(H['ath_n'].notna()).groupby(meet).transform('mean'),
})

ROUND3 = {'H2': H2, 'E': E}
for name, frame in ROUND3.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
NUMERIC_BLOCKS = {**BLOCKS, **BLOCKS2, **ROUND3}          # P T M（07）、M2 D H S（09）、H2 E（本轮）
R2_BLOCKS = ['P', 'T', 'M', 'H']                          # 09 的 ALL2
EXTRAS = {'lgb': [], 'xgb': ['M2', 'S', 'D'], 'catboost': ['M2', 'C']}   # PM：09 中各模型单独有效的块
VARIANTS3 = {'R2': [], 'H2': ['H2'], 'E': ['E'], 'PM': ['PM']}
SCREEN3 = [v for v in VARIANTS3 if RUN_ONLY is None or v in RUN_ONLY or v == 'R2']
_FRAMES3 = {}


def blocks_for(variant, model):
    out = list(R2_BLOCKS)
    for b in VARIANTS3[variant]:
        out += EXTRAS[model] if b == 'PM' else [b]
    return out


def runs(model, variant):
    """该模型在这个变体上的特征与 R2 不同才需要训练（例如 PM 对 LightGBM 没有变化）。"""
    return variant == 'R2' or blocks_for(variant, model) != blocks_for('R2', model)


def frames3(variant, model):
    """(train, test, 类别列)。"""
    blocks = blocks_for(variant, model)
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES3:
        nums = [both[RAW_NUM], F0_EXTRA] + [NUMERIC_BLOCKS[b] for b in blocks if b in NUMERIC_BLOCKS]
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} {model} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES3[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES3[key]


new_cols = pd.concat([H2, E], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print({f'{v}/{m}': frames3(v, m)[0].shape[1] for v in VARIANTS3 for m in MODELS if runs(m, v)})

## 4. 第四轮：基于成绩的特征（实验 14）

前三轮都是**不看成绩**的描述，已接近饱和。效果最大的一步（04 的同一运动员平滑）用的是同一个人在训练集里的真实成绩，但只在模型之后做线性修正。第四轮把这类信息直接做成特征：

- `AS` 运动员实力：近似同一个人其他记录的相对力量（DOTS 分）的个数、均值、最高值、时间衰减均值、年份差、体重变化，以及双胞胎的总成绩。
- `MS` 比赛实力：同一场比赛、同一组别其他选手的相对力量。

为了不泄漏，全部嵌套折外计算：外层每一折的训练部分再分 5 个内层折，每行只用其他内层折的成绩；外层验证行和测试行用整个训练部分。这些特征依赖折，所以只在训练开关打开时计算（见第 5.4 节）。

## 5. 消融实验

下表中 `mean_delta` 是五折平均的 RMSE 变化（kg，负数 = 更好），`folds_better` 是 5 折中优于基线的折数，`keep` 表示是否达到保留标准。图中柱为五折平均，黑点为单折，虚线为保留门槛 −0.05。

In [ ]:
# 只读的结果工具：读取训练时保存的 summary_*.csv 和逐折预测，不训练
def read_summary(path, index_col=0):
    path = Path(path)
    if not path.is_file():
        print(f'缺少 {path.relative_to(PROJECT_DIR)}：请确认 artifacts/ 已从 Drive 同步，或打开对应的训练开关')
        return None
    return pd.read_csv(path, index_col=index_col)


def show(table, columns=None, digits=3):
    if table is not None:
        shown = table if columns is None else table[[c for c in columns if c in table.columns]]
        display(shown.round(digits))


def plot_fold_deltas(table, baseline, title):
    """配对比较表（列 mean_delta、d_fold1–5）：柱 = 五折平均 ΔRMSE，点 = 单折，虚线 = 保留门槛 −0.05。"""
    if table is None:
        return
    rows = table.drop(index=baseline, errors='ignore').sort_values('mean_delta', ascending=False)
    folds = rows[[f'd_fold{k}' for k in range(1, N_FOLDS + 1)]].to_numpy(dtype='float64')
    colours = ['#4c72b0' if v <= KEEP_DELTA else '#c44e52' if v > 0 else '#999999' for v in rows['mean_delta']]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(rows) + 1.6))
    ax.barh(range(len(rows)), rows['mean_delta'], color=colours)
    for i in range(len(rows)):
        ax.scatter(folds[i], np.full(folds.shape[1], i), color='black', s=12, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline(KEEP_DELTA, color='#4c72b0', lw=0.8, ls='--')
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows.index.tolist())
    ax.set_xlabel(f'RMSE change vs {baseline} (kg); bar = 5-fold mean, dots = single folds')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def read_folds(directory, stem):
    """一个模型五折的 (OOF, 五折平均测试预测)；缺任何一折时返回 None。"""
    oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
    for k in range(1, N_FOLDS + 1):
        path = Path(directory) / f'{stem}__fold{k}.npz'
        if not path.is_file():
            return None
        saved = np.load(path)
        oof[saved['valid_idx']] = saved['val']
        test_pred += saved['test'] / N_FOLDS
    return (oof, test_pred) if not np.isnan(oof).any() else None

### 5.1 第一轮（实验 07，基线 F0）

三个模型都用 v2 的参数：LightGBM（CPU）、XGBoost（GPU）、CatBoost（GPU，`C1`）。实验 07 同时在最佳特征集上比较了 CatBoost 的正则化参数（结果在 03），并做了双胞胎后处理。

In [ ]:
if TRAIN['round1']:
    ensure_training_env()
    # 实验 07 的设置（结果目录：原 artifacts/feature_round1_v1）
    OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / 'round1'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RUN_PARAM_CHECK = True                 # CatBoost C2 / C3（在最佳特征集上）
    LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, cat_smooth=20, verbose=-1)
    XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                      reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                      max_cat_to_onehot=1, eval_metric='rmse')
    CB_PARAMS = {
        'C1': dict(learning_rate=0.05, depth=8, l2_leaf_reg=3.0, random_strength=1.0, bagging_temperature=1.0,
                   border_count=128),                                    # = v2
        'C2': dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                   border_count=128),                                    # 更强 L2 + 行抽样
        'C3': dict(learning_rate=0.05, depth=6, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                   border_count=128),                                    # 更浅
    }
    PATIENCE = {'lgb': 200, 'xgb': 200, 'catboost': 300}


    def fit_predict(model, key, Xa, ya, Xs, ys, Xv, Xt, rounds=MAX_ROUNDS):
        """在 (Xa, ya) 上训练、用 (Xs, ys) 早停；返回 (外层验证预测, 测试预测, 最佳轮数)。"""
        if model == 'lgb':
            m = lgb.LGBMRegressor(n_estimators=rounds, random_state=SEED, n_jobs=-1, **LGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], callbacks=[lgb.early_stopping(PATIENCE['lgb'], verbose=False)])
            best = int(m.best_iteration_ or rounds)
            return m.predict(Xv, num_iteration=best), m.predict(Xt, num_iteration=best), best
        if model == 'xgb':
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=SEED, early_stopping_rounds=PATIENCE['xgb'],
                                 **XGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
            best = int(m.best_iteration) + 1
            return (m.predict(Xv, iteration_range=(0, best)), m.predict(Xt, iteration_range=(0, best)), best)
        if model == 'catboost':
            cats = [c for c in Xa.columns if c in CAT]
            m = catboost.CatBoostRegressor(iterations=rounds, loss_function='RMSE', eval_metric='RMSE',
                                           od_type='Iter', od_wait=PATIENCE['catboost'], random_seed=SEED,
                                           verbose=0, task_type='GPU', devices='0', allow_writing_files=False,
                                           **CB_PARAMS[key])
            m.fit(catboost.Pool(Xa, ya, cat_features=cats), eval_set=catboost.Pool(Xs, ys, cat_features=cats),
                  use_best_model=True)
            best = int(m.get_best_iteration()) + 1
            return m.predict(Xv), m.predict(Xt), best
        raise ValueError(model)


    def tag_of(model, key):
        return f'catboost_{key}' if model == 'catboost' else model


    # 预检：每个（模型, 变体）在 3,000 行上训练 30 轮，几秒内暴露问题，避免跑了一小时才报错
    rng = np.random.default_rng(SEED)
    probe = rng.choice(n_train, 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    checks = [(m, v, 'C1') for v in VARIANTS for m in MODELS] + [('catboost', 'F0', 'C2'), ('catboost', 'F0', 'C3')]
    failed = {}
    for model, variant, key in checks:
        Xtr, Xte = frames_for(variant, model)
        try:
            p_val, p_test, _ = fit_predict(model, key, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                           Xte.iloc[:500], rounds=30)
            require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
            require(len(p_val) == len(pv) and len(p_test) == 500, '预测长度不符')
            print(f'预检通过：{tag_of(model, key):12s} {variant:4s}（{Xtr.shape[1]} 列）RMSE@30轮 {rmse(y[pv], p_val):.1f}')
        except Exception as exc:
            failed[(model, variant, key)] = repr(exc)
            print(f'预检失败，跳过：{tag_of(model, key)} {variant} → {exc!r}')
    require(not any(v == 'F0' and k == 'C1' for (_, v, k) in failed), '基线 F0 预检失败，请检查环境')


    CONFIG = {'purpose': 'feature round 1 (P/T/M blocks); inner 10% early stop, score early-stopped model on outer fold',
              'seed': SEED, 'n_folds': N_FOLDS, 'inner_stop_fraction': INNER_STOP_FRACTION,
              'variants': VARIANTS, 'data_sha256': EXPECTED_SHA256}
    config_path = OUTPUT_DIR / 'config.json'
    if config_path.is_file():
        require(json.loads(config_path.read_text(encoding='utf-8')) == CONFIG,
                f'{OUTPUT_DIR.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        config_path.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1), encoding='utf-8')


    def spec_of(model, variant, key):
        Xtr, _ = frames_for(variant, model)
        params = {'lgb': LGB_PARAMS, 'xgb': XGB_PARAMS, 'catboost': CB_PARAMS[key]}[model]
        return {'model': model, 'variant': variant, 'params_key': key, 'columns': list(Xtr.columns),
                'params': dict(params), 'patience': PATIENCE[model]}


    def run_one(model, variant, key, fold):
        stem = f'{tag_of(model, key)}__{variant}__fold{fold}'
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        spec = spec_of(model, variant, key)
        if json_path.is_file() and npz_path.is_file():
            saved = json.loads(json_path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID 或变体名')
            return saved, 'cache'
        Xtr, Xte = frames_for(variant, model)
        fit_idx = np.flatnonzero(FOLD_IDS != fold)
        valid_idx = np.flatnonzero(FOLD_IDS == fold)
        inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)
        started = time.perf_counter()
        p_val, p_test, best = fit_predict(model, key, Xtr.iloc[inner_fit], y[inner_fit], Xtr.iloc[inner_stop],
                                          y[inner_stop], Xtr.iloc[valid_idx], Xte)
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
        np.savez(npz_path, valid_idx=valid_idx, val=np.asarray(p_val, dtype='float64'),
                 test=np.asarray(p_test, dtype='float64'))
        result = {'tag': tag_of(model, key), 'variant': variant, 'fold': fold, 'spec': spec,
                  'n_features': int(Xtr.shape[1]), 'best_iteration': best,
                  'hit_limit': bool(best >= MAX_ROUNDS - PATIENCE[model]),
                  'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started,
                  'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        json_path.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding='utf-8')
        return result, f'{result["seconds"]:.0f}s'


    selected = [v for v in VARIANTS if RUN_ONLY is None or v in RUN_ONLY]
    for fold in range(1, N_FOLDS + 1):
        fold_started = time.perf_counter()
        for model in MODELS:
            for variant in selected:
                if (model, variant, 'C1') in failed:
                    continue
                result, how = run_one(model, variant, 'C1', fold)
                print(f'fold {fold}  {tag_of(model, "C1"):12s} {variant:4s}  RMSE {result["rmse"]:.3f}  '
                      f'最佳轮数 {result["best_iteration"]:5d}  ({how})')
        print(f'—— fold {fold} 完成，本次运行用时 {(time.perf_counter() - fold_started) / 60:.1f} 分钟')


    def load_results():
        rows = [json.loads(p.read_text(encoding='utf-8')) for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))]
        require(rows, '还没有完成的结果')
        return pd.DataFrame(rows)


    def load_oof(tag, variant):
        """五折都完成时返回 (OOF, 五折平均测试预测)，否则 (None, None)。"""
        oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
        for fold in range(1, N_FOLDS + 1):
            path = OUTPUT_DIR / f'{tag}__{variant}__fold{fold}.npz'
            if not path.is_file():
                return None, None
            saved = np.load(path)
            oof[saved['valid_idx']] = saved['val']
            test_pred += saved['test'] / N_FOLDS
        require(not np.isnan(oof).any(), f'{tag} {variant} 的 OOF 不完整')
        return oof, test_pred


    def fold_rmse(pred):
        return pd.Series({k: rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)})


    def paired_table(preds, base='F0'):
        """preds: {变体: OOF}；与 base 在同一五折上配对比较。"""
        require(base in preds, f'缺少基线 {base}')
        base_folds = fold_rmse(preds[base])
        rows = []
        for name, pred in preds.items():
            delta = fold_rmse(pred) - base_folds
            rows.append({'variant': name, 'oof_rmse': rmse(y, pred), 'mean_delta': delta.mean(),
                         'folds_better': int((delta < 0).sum()), **{f'd_fold{k}': v for k, v in delta.items()}})
        table = pd.DataFrame(rows).set_index('variant')
        table['keep'] = (table['mean_delta'] <= KEEP_DELTA) & (table['folds_better'] >= 4)
        table.loc[base, 'keep'] = False
        return table.sort_values('oof_rmse')


    results = load_results()
    SINGLE, TESTS = {}, {}
    for tag in ['lgb', 'xgb', 'catboost_C1']:
        preds = {}
        for variant in VARIANTS:
            oof, test_pred = load_oof(tag, variant)
            if oof is not None:
                preds[variant], TESTS[(tag, variant)] = oof, test_pred
        if 'F0' in preds:
            SINGLE[tag] = paired_table(preds)
            part = results[results['tag'] == tag].groupby('variant')
            SINGLE[tag]['mean_best_iter'] = part['best_iteration'].mean()
            SINGLE[tag]['minutes'] = part['seconds'].sum() / 60
            print(f'\n=== {tag}')
            display(SINGLE[tag].round(3))

    BLENDS = {}
    for variant in VARIANTS:
        if all((tag, variant) in TESTS for tag in ['lgb', 'xgb', 'catboost_C1']):
            P_oof = np.column_stack([load_oof(tag, variant)[0] for tag in ['lgb', 'xgb', 'catboost_C1']])
            P_test = np.column_stack([TESTS[(tag, variant)] for tag in ['lgb', 'xgb', 'catboost_C1']])
            w, _ = nnls(P_oof, y)
            BLENDS[variant] = {'oof': P_oof @ w, 'test': P_test @ w, 'weights': w.round(3).tolist()}
    require('F0' in BLENDS, '三模型的 F0 还没有全部完成')
    blend_table = paired_table({v: b['oof'] for v, b in BLENDS.items()})
    blend_table['weights_lgb_xgb_cb'] = [str(BLENDS[v]['weights']) for v in blend_table.index]
    print('\n=== 三模型 NNLS 融合（主要判读依据）')
    display(blend_table.round(3))
    BEST = blend_table.index[0]
    print(f'融合 OOF 最好的变体：{BEST}；达到保留标准的变体：{blend_table.index[blend_table["keep"]].tolist() or "无"}')

    pd.concat({tag: t for tag, t in SINGLE.items()}, names=['model']).to_csv(OUTPUT_DIR / 'summary_single.csv')
    blend_table.to_csv(OUTPUT_DIR / 'summary_blend.csv')


    def residual_table(pred):
        res = y - pred
        groups = {
            'age_half': P['age_half'].iloc[:n_train].to_numpy(),
            'bw_precision': P['bw_precision'].iloc[:n_train].to_numpy(),
            'other_state(-1=未知)': T['other_state'].iloc[:n_train].fillna(-1).to_numpy(),
            'abroad(-1=未知)': T['abroad'].iloc[:n_train].fillna(-1).to_numpy(),
            'meet_n 八分位': pd.qcut(M['meet_n'].iloc[:n_train], 8, labels=False, duplicates='drop').to_numpy(),
        }
        frames = []
        for name, key in groups.items():
            g = pd.DataFrame({'k': key, 'r': res}).groupby('k')['r'].agg(['size', 'mean'])
            g.index = pd.MultiIndex.from_product([[name], g.index], names=['signal', 'level'])
            frames.append(g)
        return pd.concat(frames)


    check = residual_table(BLENDS['F0']['oof']).rename(columns={'mean': 'F0'})
    check[BEST] = residual_table(BLENDS[BEST]['oof'])['mean']
    display(check.round(2))


    if RUN_PARAM_CHECK:
        for fold in range(1, N_FOLDS + 1):
            for key in ['C2', 'C3']:
                if ('catboost', 'F0', key) in failed:
                    continue
                result, how = run_one('catboost', BEST, key, fold)
                print(f'fold {fold}  catboost_{key}  {BEST:4s}  RMSE {result["rmse"]:.3f}  '
                      f'最佳轮数 {result["best_iteration"]:5d}  ({how})')
        preds = {}
        for key in ['C1', 'C2', 'C3']:
            oof, test_pred = load_oof(f'catboost_{key}', BEST)
            if oof is not None:
                preds[key], TESTS[(f'catboost_{key}', BEST)] = oof, test_pred
        param_table = paired_table(preds, base='C1')
        display(param_table.round(3))
        param_table.to_csv(OUTPUT_DIR / 'summary_catboost_params.csv')
        BEST_CB = param_table.index[0] if param_table.loc[param_table.index[0], 'keep'] else 'C1'
    else:
        BEST_CB = 'C1'
    tags = ['lgb', 'xgb', f'catboost_{BEST_CB}']
    P_oof = np.column_stack([load_oof(tag, BEST)[0] for tag in tags])
    P_test = np.column_stack([TESTS[(tag, BEST)] for tag in tags])
    w, _ = nnls(P_oof, y)
    FINAL = {'name': f'{BEST}_blend3_{BEST_CB}', 'oof': P_oof @ w, 'test': P_test @ w}
    print(f'最终组合：特征 {BEST}，CatBoost {BEST_CB}，权重 {dict(zip(tags, w.round(3)))}，'
          f'OOF RMSE {rmse(y, FINAL["oof"]):.3f}（F0 融合 {rmse(y, BLENDS["F0"]["oof"]):.3f}）')


    CORE = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                       # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                        # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(train_keys_exact, train_keys_core, query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(train_keys_exact, query_exact)
        n_co, m_co = mate_stats(train_keys_core, query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / np.sum(d * d), 0, 1)) if mask.any() else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    ex_tr, co_tr = row_keys(train, RAW_FEATURES), row_keys(train, CORE)
    ex_te, co_te = row_keys(test, RAW_FEATURES), row_keys(test, CORE)
    LEVEL, MATE = twins(ex_tr, co_tr)
    T_LEVEL, T_MATE = twins(ex_tr, co_tr, ex_te, co_te)
    half = (co_tr % 2).astype(int)

    twin_rows, exports = [], {'F0_blend3': BLENDS['F0'], FINAL['name']: FINAL}
    for name, item in exports.items():
        oof = item['oof']
        alphas = {lv: fit_alpha(LEVEL == lv, oof, MATE) for lv in ['exact', 'core']}
        crossfit = oof.copy()
        for h in (0, 1):
            a = {lv: fit_alpha((half != h) & (LEVEL == lv), oof, MATE) for lv in ['exact', 'core']}
            crossfit[half == h] = apply_twins(oof[half == h], MATE[half == h], LEVEL[half == h], a)
        test_fixed = apply_twins(item['test'], T_MATE, T_LEVEL, alphas)
        twin_rows.append({'name': name, 'oof_rmse': rmse(y, oof), 'oof_rmse_twinfix_crossfit': rmse(y, crossfit),
                          'alpha_exact': alphas['exact'], 'alpha_core': alphas['core'],
                          'test_rows_exact': int((T_LEVEL == 'exact').sum()), 'test_rows_core': int((T_LEVEL == 'core').sum())})
        for suffix, pred in [('', item['test']), ('_twinfix', test_fixed)]:
            path = OUTPUT_DIR / f'submission_{name}{suffix}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(pred, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
    twin_table = pd.DataFrame(twin_rows).set_index('name')
    display(twin_table.round(3))
    twin_table.to_csv(OUTPUT_DIR / 'summary_twins.csv')
    np.save(OUTPUT_DIR / 'oof_F0_blend3.npy', BLENDS['F0']['oof'])
    np.save(OUTPUT_DIR / f'oof_{FINAL["name"]}.npy', FINAL['oof'])
    print('完成。')
else:
    print("训练开关关闭（TRAIN['round1'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_round1():
    directory = ARTIFACTS / '02_feature_engineering' / 'round1'
    single = read_summary(directory / 'summary_single.csv', index_col=[0, 1])
    if single is not None:
        print('单模型五折 OOF RMSE（kg）：')
        display(single['oof_rmse'].unstack(0).reindex(['F0', 'T', 'P', 'PT', 'M', 'ALL']).round(3))
    blend = read_summary(directory / 'summary_blend.csv')
    print('三模型 NNLS 融合（主要判读依据）：')
    show(blend, ['oof_rmse', 'mean_delta', 'folds_better', 'keep', 'weights_lgb_xgb_cb'])
    plot_fold_deltas(blend, 'F0', 'Round 1: three-model blend vs F0')
    print('双胞胎后处理（alpha 按核心键分两半交叉拟合）：')
    show(read_summary(directory / 'summary_twins.csv'))


show_round1()

**结论**：
- `P` 记录精度 −0.33、`M` 比赛背景 −0.36，都是 5/5 折；`T` 跨地参赛 −0.04，未达标，但它的列被 M、H2 使用。
- **ALL = P + T + M 合计 −0.67（5/5 折）**，三个模型方向一致。
- 双胞胎后处理再降约 0.25。
- 公开榜：F0 + 双胞胎 77.947 → ALL + CatBoost C2 + 双胞胎 77.158，与交叉验证一致。

### 5.2 第二轮（实验 09，基线 R1 = F0 + P + T + M）

CatBoost 改用 03 选出的 C2 参数（L2 10、Bernoulli 0.8）。

In [ ]:
if TRAIN['round2']:
    ensure_training_env()
    # 实验 09 的设置（结果目录：原 artifacts/feature_round2_v1）
    OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / 'round2'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, cat_smooth=20, verbose=-1)
    XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                      reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                      max_cat_to_onehot=1, eval_metric='rmse')
    CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                     border_count=128)                                   # 07 的 C2
    PATIENCE = {'lgb': 200, 'xgb': 200, 'catboost': 300}


    def fit_predict(model, Xa, ya, Xs, ys, Xv, Xt, cat_cols, rounds=MAX_ROUNDS):
        """在 (Xa, ya) 上训练、用 (Xs, ys) 早停；返回 (外层验证预测, 测试预测, 最佳轮数)。"""
        if model == 'lgb':
            m = lgb.LGBMRegressor(n_estimators=rounds, random_state=SEED, n_jobs=-1, **LGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], callbacks=[lgb.early_stopping(PATIENCE['lgb'], verbose=False)])
            best = int(m.best_iteration_ or rounds)
            return m.predict(Xv, num_iteration=best), m.predict(Xt, num_iteration=best), best
        if model == 'xgb':
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=SEED, early_stopping_rounds=PATIENCE['xgb'],
                                 **XGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
            best = int(m.best_iteration) + 1
            return m.predict(Xv, iteration_range=(0, best)), m.predict(Xt, iteration_range=(0, best)), best
        if model == 'catboost':
            m = catboost.CatBoostRegressor(iterations=rounds, loss_function='RMSE', eval_metric='RMSE',
                                           od_type='Iter', od_wait=PATIENCE['catboost'], random_seed=SEED,
                                           verbose=0, task_type='GPU', devices='0', allow_writing_files=False,
                                           **CB_PARAMS)
            m.fit(catboost.Pool(Xa, ya, cat_features=cat_cols), eval_set=catboost.Pool(Xs, ys, cat_features=cat_cols),
                  use_best_model=True)
            best = int(m.get_best_iteration()) + 1
            return m.predict(Xv), m.predict(Xt), best
        raise ValueError(model)


    rng = np.random.default_rng(SEED)
    probe = rng.choice(n_train, 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    failed = {}
    for variant in SCREEN:
        for model in models_for(variant):
            Xtr, Xte, cat_cols = frames2(variant, model)
            try:
                p_val, p_test, _ = fit_predict(model, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                               Xte.iloc[:500], cat_cols, rounds=30)
                require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
                require(len(p_val) == len(pv) and len(p_test) == 500, '预测长度不符')
                print(f'预检通过：{model:8s} {variant:3s}（{Xtr.shape[1]} 列）RMSE@30轮 {rmse(y[pv], p_val):.1f}')
            except Exception as exc:
                failed[(model, variant)] = repr(exc)
                print(f'预检失败，跳过：{model} {variant} → {exc!r}')
    require(not any(v == 'R1' for (_, v) in failed), '基线 R1 预检失败，请检查环境')


    CONFIG = {'purpose': 'feature round 2 on top of 07 ALL; inner 10% early stop, score early-stopped model on outer fold',
              'seed': SEED, 'n_folds': N_FOLDS, 'inner_stop_fraction': INNER_STOP_FRACTION,
              'r1_blocks': R1_BLOCKS, 'screen_blocks': {k: v for k, v in VARIANT_BLOCKS.items()},
              'data_sha256': EXPECTED_SHA256}
    config_path = OUTPUT_DIR / 'config.json'
    if config_path.is_file():
        require(json.loads(config_path.read_text(encoding='utf-8')) == CONFIG,
                f'{OUTPUT_DIR.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        config_path.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1), encoding='utf-8')


    def spec_of(model, variant):
        Xtr, _, cat_cols = frames2(variant, model)
        params = {'lgb': LGB_PARAMS, 'xgb': XGB_PARAMS, 'catboost': CB_PARAMS}[model]
        return {'model': model, 'variant': variant, 'blocks': list(VARIANT_BLOCKS[variant]),
                'columns': list(Xtr.columns), 'cat_cols': list(cat_cols), 'params': dict(params),
                'patience': PATIENCE[model]}


    def run_one(model, variant, fold):
        stem = f'{model}__{variant}__fold{fold}'
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        spec = spec_of(model, variant)
        if json_path.is_file() and npz_path.is_file():
            saved = json.loads(json_path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID 或变体名')
            return saved, 'cache'
        Xtr, Xte, cat_cols = frames2(variant, model)
        fit_idx = np.flatnonzero(FOLD_IDS != fold)
        valid_idx = np.flatnonzero(FOLD_IDS == fold)
        inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 07
        started = time.perf_counter()
        p_val, p_test, best = fit_predict(model, Xtr.iloc[inner_fit], y[inner_fit], Xtr.iloc[inner_stop],
                                          y[inner_stop], Xtr.iloc[valid_idx], Xte, cat_cols)
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
        np.savez(npz_path, valid_idx=valid_idx, val=np.asarray(p_val, dtype='float64'),
                 test=np.asarray(p_test, dtype='float64'))
        result = {'model': model, 'variant': variant, 'fold': fold, 'spec': spec, 'n_features': int(Xtr.shape[1]),
                  'best_iteration': best, 'hit_limit': bool(best >= MAX_ROUNDS - PATIENCE[model]),
                  'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started,
                  'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        json_path.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding='utf-8')
        return result, f'{result["seconds"]:.0f}s'


    def run_variants(variants):
        for fold in range(1, N_FOLDS + 1):
            fold_started = time.perf_counter()
            for model in MODELS:
                for variant in variants:
                    if model not in models_for(variant) or (model, variant) in failed:
                        continue
                    result, how = run_one(model, variant, fold)
                    print(f'fold {fold}  {model:8s} {variant:4s}  RMSE {result["rmse"]:.3f}  '
                          f'最佳轮数 {result["best_iteration"]:5d}  ({how})')
            print(f'—— fold {fold} 完成，本次运行用时 {(time.perf_counter() - fold_started) / 60:.1f} 分钟')


    run_variants(SCREEN)


    def load_oof(model, variant):
        """五折都完成时返回 (OOF, 五折平均测试预测)，否则 None。"""
        oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
        for fold in range(1, N_FOLDS + 1):
            path = OUTPUT_DIR / f'{model}__{variant}__fold{fold}.npz'
            if not path.is_file():
                return None
            saved = np.load(path)
            oof[saved['valid_idx']] = saved['val']
            test_pred += saved['test'] / N_FOLDS
        require(not np.isnan(oof).any(), f'{model} {variant} 的 OOF 不完整')
        return oof, test_pred


    def fold_rmse(pred):
        return pd.Series({k: rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)})


    def paired_table(preds, base='R1'):
        require(base in preds, f'缺少基线 {base}')
        base_folds = fold_rmse(preds[base])
        rows = []
        for name, pred in preds.items():
            delta = fold_rmse(pred) - base_folds
            rows.append({'variant': name, 'oof_rmse': rmse(y, pred), 'mean_delta': delta.mean(),
                         'folds_better': int((delta < 0).sum()), **{f'd_fold{k}': v for k, v in delta.items()}})
        table = pd.DataFrame(rows).set_index('variant')
        table['keep'] = (table['mean_delta'] <= KEEP_DELTA) & (table['folds_better'] >= 4)
        table.loc[base, 'keep'] = False
        return table.sort_values('oof_rmse')


    def blend_variant(variant):
        """三模型 NNLS 融合；某模型不跑该变体时（如 C 只改 CatBoost）用它的 R1 结果。"""
        loaded = []
        for model in MODELS:
            item = load_oof(model, variant if model in models_for(variant) else 'R1')
            if item is None:
                return None
            loaded.append(item)
        P_oof = np.column_stack([o for o, _ in loaded])
        P_test = np.column_stack([t for _, t in loaded])
        w, _ = nnls(P_oof, y)
        return {'oof': P_oof @ w, 'test': P_test @ w, 'weights': [round(float(v), 3) for v in w]}


    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
    for model in MODELS:
        preds = {}
        for variant in VARIANT_BLOCKS:
            item = load_oof(model, variant) if model in models_for(variant) else None
            if item is not None:
                preds[variant] = item[0]
        if 'R1' in preds:
            table = paired_table(preds)
            part = results[results['model'] == model].groupby('variant')
            table['mean_best_iter'] = part['best_iteration'].mean()
            table['minutes'] = part['seconds'].sum() / 60
            print(f'\n=== {model}')
            display(table.round(3))
            table.to_csv(OUTPUT_DIR / f'summary_single_{model}.csv')

    BLENDS = {}
    for variant in VARIANT_BLOCKS:
        item = blend_variant(variant)
        if item is not None:
            BLENDS[variant] = item
    require('R1' in BLENDS, '三模型的 R1 还没有全部完成')
    blend_table = paired_table({v: b['oof'] for v, b in BLENDS.items()})
    blend_table['weights_lgb_xgb_cb'] = [str(BLENDS[v]['weights']) for v in blend_table.index]
    print('\n=== 三模型 NNLS 融合（主要判读依据）')
    display(blend_table.round(3))
    blend_table.to_csv(OUTPUT_DIR / 'summary_blend_screen.csv')

    KEPT = [v for v in ['M2', 'D', 'H', 'S', 'C'] if v in blend_table.index and bool(blend_table.loc[v, 'keep'])]
    VARIANT_BLOCKS['ALL2'] = KEPT
    MODELS_FOR['ALL2'] = ['catboost'] if KEPT and all(b in CB_ONLY for b in KEPT) else MODELS
    print('达标的新块：', KEPT or '无', '→ ALL2 =', ['R1'] + KEPT)


    if KEPT:
        for model in models_for('ALL2'):
            Xtr, Xte, cat_cols = frames2('ALL2', model)
            p_val, p_test, _ = fit_predict(model, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                           Xte.iloc[:500], cat_cols, rounds=30)          # ALL2 的预检
            require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'ALL2 {model} 预检失败')
        run_variants(['ALL2'])
        BLENDS['ALL2'] = blend_variant('ALL2')
        final_table = paired_table({v: BLENDS[v]['oof'] for v in ['R1'] + KEPT + ['ALL2']})
        display(final_table.round(3))
        final_table.to_csv(OUTPUT_DIR / 'summary_blend_all2.csv')
        FINAL = 'ALL2'
    else:
        print('没有达标的新块，保持 R1。')
        FINAL = 'R1'

    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / np.sum(d * d), 0, 1)) if mask.any() else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    half = (row_keys(train, CORE) % 2).astype(int)
    twin_rows = []
    for name in ['R1', FINAL] if FINAL != 'R1' else ['R1']:
        oof, test_pred = BLENDS[name]['oof'], BLENDS[name]['test']
        alphas = {lv: fit_alpha(LEVEL == lv, oof, MATE) for lv in ['exact', 'core']}
        crossfit = oof.copy()
        for h in (0, 1):
            a = {lv: fit_alpha((half != h) & (LEVEL == lv), oof, MATE) for lv in ['exact', 'core']}
            crossfit[half == h] = apply_twins(oof[half == h], MATE[half == h], LEVEL[half == h], a)
        fixed = apply_twins(test_pred, T_MATE, T_LEVEL, alphas)
        twin_rows.append({'name': name, 'oof_rmse': rmse(y, oof), 'oof_rmse_twinfix_crossfit': rmse(y, crossfit),
                          'alpha_exact': alphas['exact'], 'alpha_core': alphas['core']})
        for suffix, pred in [('', test_pred), ('_twinfix', fixed)]:
            path = OUTPUT_DIR / f'submission_{name}_blend3{suffix}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(pred, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
        np.save(OUTPUT_DIR / f'oof_{name}_blend3.npy', oof)
    twin_table = pd.DataFrame(twin_rows).set_index('name')
    display(twin_table.round(3))
    twin_table.to_csv(OUTPUT_DIR / 'summary_twins.csv')


    def bucket(values, edges, labels):
        return pd.cut(values, edges, labels=labels, right=False).astype(str)


    groups = {
        'age_half': P['age_half'].iloc[:n_train].to_numpy(),
        'bw_precision': P['bw_precision'].iloc[:n_train].to_numpy(),
        'meet_n 八分位': pd.qcut(M['meet_n'].iloc[:n_train], 8, labels=False, duplicates='drop').to_numpy(),
        'meet_n_states': bucket(M2['meet_n_states'].iloc[:n_train], [0, 1, 2, 5, 10, 1000], ['0', '1', '2-4', '5-9', '10+']),
        'ath_n': bucket(H['ath_n'].iloc[:n_train].fillna(-1), [-1, 0, 2, 3, 6, 1e9], ['未知', '1', '2', '3-5', '6+']),
        'ath_years_since_first': bucket(H['ath_years_since_first'].iloc[:n_train].fillna(-1), [-1, 0, 1, 2, 5, 100],
                                        ['未知', '0', '1', '2-4', '5+']),
        'dup_exact_n>0': (D['dup_exact_n'].iloc[:n_train] > 0).to_numpy(),
        'div_n': bucket(D['div_n'].iloc[:n_train], [1, 2, 4, 8, 1e9], ['1', '2-3', '4-7', '8+']),
        'Sex×Equipment': (train['Sex'] + '|' + train['Equipment']).to_numpy(),
        'Tested': train['Tested'].to_numpy(),
        'MeetYear 段': bucket(train['MeetYear'], [1900, 2000, 2010, 2015, 2020, 2030],
                              ['<2000', '2000s', '2010-14', '2015-19', '2020+']),
    }
    frames = []
    for name in ['R1', FINAL] if FINAL != 'R1' else ['R1']:
        res = y - BLENDS[name]['oof']
        rows = []
        for g, key in groups.items():
            part = pd.DataFrame({'k': key, 'r': res}).groupby('k')['r'].agg(['size', 'mean'])
            part.index = pd.MultiIndex.from_product([[g], part.index.astype(str)], names=['signal', 'level'])
            rows.append(part)
        frames.append(pd.concat(rows).rename(columns={'mean': name}))
    diag = pd.concat(frames, axis=1)
    diag = diag.loc[:, ~diag.columns.duplicated()]
    display(diag.round(2))
    diag.to_csv(OUTPUT_DIR / 'diagnostics_residuals.csv')
    print('完成。')
else:
    print("训练开关关闭（TRAIN['round2'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_round2():
    directory = ARTIFACTS / '02_feature_engineering' / 'round2'
    singles = {m: read_summary(directory / f'summary_single_{m}.csv') for m in ['lgb', 'xgb', 'catboost']}
    singles = {m: t for m, t in singles.items() if t is not None}
    if singles:
        print('单模型：相对 R1 的平均 ΔRMSE（kg，负数 = 更好；C 只给 CatBoost）：')
        display(pd.DataFrame({m: t['mean_delta'] for m, t in singles.items()})
                .reindex(['M2', 'D', 'H', 'S', 'C']).round(3))
    screen = read_summary(directory / 'summary_blend_screen.csv')
    print('三模型 NNLS 融合：')
    show(screen, ['oof_rmse', 'mean_delta', 'folds_better', 'keep', 'weights_lgb_xgb_cb'])
    plot_fold_deltas(screen, 'R1', 'Round 2: three-model blend vs R1')
    print('达标的块合成 ALL2：')
    show(read_summary(directory / 'summary_blend_all2.csv'), ['oof_rmse', 'mean_delta', 'folds_better', 'keep'])


show_round2()

**结论**：
- 只有 **`H` 运动员历史达标（−0.13，5/5 折）**，ALL2 = R1 + H。
- `C` 让 CatBoost 单独 −0.12，但融合只 −0.05。
- `M2` 对 CatBoost、XGBoost 各约 −0.06，对 LightGBM +0.07；`D`、`S` 只对 XGBoost 有效。
- 这些「对单个模型有效」的块在第三轮的 `PM` 中按模型加入。

### 5.3 第三轮（实验 10，基线 R2 = ALL2）

实验 10 还在 ALL3 融合上比较了 5 种第二层平滑的做法，结果在 05。

In [ ]:
if TRAIN['round3']:
    ensure_training_env()
    # 实验 10 的设置（结果目录：原 artifacts/feature_round3_v1）
    OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / 'round3'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, cat_smooth=20, verbose=-1)
    XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                      reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                      max_cat_to_onehot=1, eval_metric='rmse')
    CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                     border_count=128)                                   # 07 的 C2
    PATIENCE = {'lgb': 200, 'xgb': 200, 'catboost': 300}


    def fit_predict(model, Xa, ya, Xs, ys, Xv, Xt, cat_cols, rounds=MAX_ROUNDS):
        """在 (Xa, ya) 上训练、用 (Xs, ys) 早停；返回 (外层验证预测, 测试预测, 最佳轮数)。"""
        if model == 'lgb':
            m = lgb.LGBMRegressor(n_estimators=rounds, random_state=SEED, n_jobs=-1, **LGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], callbacks=[lgb.early_stopping(PATIENCE['lgb'], verbose=False)])
            best = int(m.best_iteration_ or rounds)
            return m.predict(Xv, num_iteration=best), m.predict(Xt, num_iteration=best), best
        if model == 'xgb':
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=SEED, early_stopping_rounds=PATIENCE['xgb'],
                                 **XGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
            best = int(m.best_iteration) + 1
            return m.predict(Xv, iteration_range=(0, best)), m.predict(Xt, iteration_range=(0, best)), best
        if model == 'catboost':
            m = catboost.CatBoostRegressor(iterations=rounds, loss_function='RMSE', eval_metric='RMSE',
                                           od_type='Iter', od_wait=PATIENCE['catboost'], random_seed=SEED,
                                           verbose=0, task_type='GPU', devices='0', allow_writing_files=False,
                                           **CB_PARAMS)
            m.fit(catboost.Pool(Xa, ya, cat_features=cat_cols), eval_set=catboost.Pool(Xs, ys, cat_features=cat_cols),
                  use_best_model=True)
            best = int(m.get_best_iteration()) + 1
            return m.predict(Xv), m.predict(Xt), best
        raise ValueError(model)



    rng = np.random.default_rng(SEED)
    probe = rng.choice(n_train, 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    failed = {}
    for variant in SCREEN3:
        for model in MODELS:
            if not runs(model, variant):
                continue
            Xtr, Xte, cat_cols = frames3(variant, model)
            try:
                p_val, p_test, _ = fit_predict(model, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                               Xte.iloc[:500], cat_cols, rounds=30)
                require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
                require(len(p_val) == len(pv) and len(p_test) == 500, '预测长度不符')
                print(f'预检通过：{model:8s} {variant:3s}（{Xtr.shape[1]} 列）RMSE@30轮 {rmse(y[pv], p_val):.1f}')
            except Exception as exc:
                failed[(model, variant)] = repr(exc)
                print(f'预检失败，跳过：{model} {variant} → {exc!r}')
    require(not any(v == 'R2' for (_, v) in failed), '基线 R2 预检失败，请检查环境')


    CONFIG = {'purpose': 'feature round 3 on top of 09 ALL2; inner 10% early stop, score early-stopped model on outer fold',
              'seed': SEED, 'n_folds': N_FOLDS, 'inner_stop_fraction': INNER_STOP_FRACTION,
              'r2_blocks': R2_BLOCKS, 'extras': EXTRAS, 'screen': VARIANTS3, 'data_sha256': EXPECTED_SHA256}
    config_path = OUTPUT_DIR / 'config.json'
    if config_path.is_file():
        require(json.loads(config_path.read_text(encoding='utf-8')) == CONFIG,
                f'{OUTPUT_DIR.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        config_path.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1), encoding='utf-8')


    def spec_of(model, variant):
        Xtr, _, cat_cols = frames3(variant, model)
        params = {'lgb': LGB_PARAMS, 'xgb': XGB_PARAMS, 'catboost': CB_PARAMS}[model]
        return {'model': model, 'variant': variant, 'blocks': blocks_for(variant, model),
                'columns': list(Xtr.columns), 'cat_cols': list(cat_cols), 'params': dict(params),
                'patience': PATIENCE[model]}


    def run_one(model, variant, fold):
        stem = f'{model}__{variant}__fold{fold}'
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        spec = spec_of(model, variant)
        if json_path.is_file() and npz_path.is_file():
            saved = json.loads(json_path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID 或变体名')
            return saved, 'cache'
        Xtr, Xte, cat_cols = frames3(variant, model)
        fit_idx = np.flatnonzero(FOLD_IDS != fold)
        valid_idx = np.flatnonzero(FOLD_IDS == fold)
        inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 07
        started = time.perf_counter()
        p_val, p_test, best = fit_predict(model, Xtr.iloc[inner_fit], y[inner_fit], Xtr.iloc[inner_stop],
                                          y[inner_stop], Xtr.iloc[valid_idx], Xte, cat_cols)
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
        np.savez(npz_path, valid_idx=valid_idx, val=np.asarray(p_val, dtype='float64'),
                 test=np.asarray(p_test, dtype='float64'))
        result = {'model': model, 'variant': variant, 'fold': fold, 'spec': spec, 'n_features': int(Xtr.shape[1]),
                  'best_iteration': best, 'hit_limit': bool(best >= MAX_ROUNDS - PATIENCE[model]),
                  'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started,
                  'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        json_path.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding='utf-8')
        return result, f'{result["seconds"]:.0f}s'


    def run_variants(variants):
        for fold in range(1, N_FOLDS + 1):
            fold_started = time.perf_counter()
            for model in MODELS:
                for variant in variants:
                    if not runs(model, variant) or (model, variant) in failed:
                        continue
                    result, how = run_one(model, variant, fold)
                    print(f'fold {fold}  {model:8s} {variant:4s}  RMSE {result["rmse"]:.3f}  '
                          f'最佳轮数 {result["best_iteration"]:5d}  ({how})')
            print(f'—— fold {fold} 完成，本次运行用时 {(time.perf_counter() - fold_started) / 60:.1f} 分钟')


    run_variants(SCREEN3)


    def load_oof(model, variant):
        """五折都完成时返回 (OOF, 五折平均测试预测)，否则 None。"""
        oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
        for fold in range(1, N_FOLDS + 1):
            path = OUTPUT_DIR / f'{model}__{variant}__fold{fold}.npz'
            if not path.is_file():
                return None
            saved = np.load(path)
            oof[saved['valid_idx']] = saved['val']
            test_pred += saved['test'] / N_FOLDS
        require(not np.isnan(oof).any(), f'{model} {variant} 的 OOF 不完整')
        return oof, test_pred


    def fold_rmse(pred):
        return pd.Series({k: rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)})


    def paired_table(preds, base='R2'):
        require(base in preds, f'缺少基线 {base}')
        base_folds = fold_rmse(preds[base])
        rows = []
        for name, pred in preds.items():
            delta = fold_rmse(pred) - base_folds
            rows.append({'variant': name, 'oof_rmse': rmse(y, pred), 'mean_delta': delta.mean(),
                         'folds_better': int((delta < 0).sum()), **{f'd_fold{k}': v for k, v in delta.items()}})
        table = pd.DataFrame(rows).set_index('variant')
        table['keep'] = (table['mean_delta'] <= KEEP_DELTA) & (table['folds_better'] >= 4)
        table.loc[base, 'keep'] = False
        return table.sort_values('oof_rmse')


    def blend_variant(variant):
        """三模型 NNLS 融合；某模型在该变体上特征不变时用它的 R2 结果。"""
        loaded = []
        for model in MODELS:
            item = load_oof(model, variant if runs(model, variant) else 'R2')
            if item is None:
                return None
            loaded.append(item)
        P_oof = np.column_stack([o for o, _ in loaded])
        P_test = np.column_stack([t for _, t in loaded])
        w, _ = nnls(P_oof, y)
        return {'oof': P_oof @ w, 'test': P_test @ w, 'weights': [round(float(v), 3) for v in w]}


    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
    for model in MODELS:
        preds = {}
        for variant in VARIANTS3:
            item = load_oof(model, variant) if runs(model, variant) else None
            if item is not None:
                preds[variant] = item[0]
        if 'R2' in preds:
            table = paired_table(preds)
            part = results[results['model'] == model].groupby('variant')
            table['mean_best_iter'] = part['best_iteration'].mean()
            table['minutes'] = part['seconds'].sum() / 60
            print(f'\n=== {model}')
            display(table.round(3))
            table.to_csv(OUTPUT_DIR / f'summary_single_{model}.csv')

    BLENDS = {}
    for variant in VARIANTS3:
        item = blend_variant(variant)
        if item is not None:
            BLENDS[variant] = item
    require('R2' in BLENDS, '三模型的 R2 还没有全部完成')
    blend_table = paired_table({v: b['oof'] for v, b in BLENDS.items()})
    blend_table['weights_lgb_xgb_cb'] = [str(BLENDS[v]['weights']) for v in blend_table.index]
    print('\n=== 三模型 NNLS 融合（主要判读依据）')
    display(blend_table.round(3))
    blend_table.to_csv(OUTPUT_DIR / 'summary_blend_screen.csv')

    KEPT = [v for v in ['H2', 'E', 'PM'] if v in blend_table.index and bool(blend_table.loc[v, 'keep'])]
    VARIANTS3['ALL3'] = KEPT
    print('达标的新块：', KEPT or '无', '→ ALL3 = R2 +', KEPT)


    if KEPT:
        for model in MODELS:
            if runs(model, 'ALL3'):
                Xtr, Xte, cat_cols = frames3('ALL3', model)
                p_val, p_test, _ = fit_predict(model, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                               Xte.iloc[:500], cat_cols, rounds=30)          # ALL3 的预检
                require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'ALL3 {model} 预检失败')
        run_variants(['ALL3'])
        BLENDS['ALL3'] = blend_variant('ALL3')
        final_table = paired_table({v: BLENDS[v]['oof'] for v in ['R2'] + KEPT + ['ALL3']})
        display(final_table.round(3))
        final_table.to_csv(OUTPUT_DIR / 'summary_blend_all3.csv')
        FINAL = 'ALL3'
    else:
        print('没有达标的新块，保持 R2。')
        FINAL = 'R2'

    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        denom = float(np.sum(d * d))
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
    twin_rows = []
    for name in (['R2', FINAL] if FINAL != 'R2' else ['R2']):
        oof, test_pred = BLENDS[name]['oof'], BLENDS[name]['test']
        alphas = {lv: fit_alpha(LEVEL == lv, oof, MATE) for lv in ['exact', 'core']}
        crossfit = oof.copy()
        for h in (0, 1):
            a = {lv: fit_alpha((HALF != h) & (LEVEL == lv), oof, MATE) for lv in ['exact', 'core']}
            crossfit[HALF == h] = apply_twins(oof[HALF == h], MATE[HALF == h], LEVEL[HALF == h], a)
        fixed = apply_twins(test_pred, T_MATE, T_LEVEL, alphas)
        twin_rows.append({'name': name, 'oof_rmse': rmse(y, oof), 'oof_rmse_twinfix_crossfit': rmse(y, crossfit),
                          'alpha_exact': alphas['exact'], 'alpha_core': alphas['core']})
        for suffix, pred in [('', test_pred), ('_twinfix', fixed)]:
            path = OUTPUT_DIR / f'submission_{name}_blend3{suffix}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(pred, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
        np.save(OUTPUT_DIR / f'oof_{name}_blend3.npy', oof)
        np.save(OUTPUT_DIR / f'test_{name}_blend3.npy', test_pred)
    twin_table = pd.DataFrame(twin_rows).set_index('name')
    display(twin_table.round(3))
    twin_table.to_csv(OUTPUT_DIR / 'summary_twins.csv')


    base_oof, base_test = BLENDS[FINAL]['oof'], BLENDS[FINAL]['test']
    TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


    def text_key(frame, columns):
        out = frame[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + frame[c].astype(str)
        return out


    BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
    BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
    YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
    PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
              'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
              'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
    SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
    PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
    PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
    MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
    MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
    VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
    VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


    def sums(keys, values):
        table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
        return table['sum'], table['count']


    def look(table, keys):
        s, n = table
        return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


    def f_exact(k, r):
        ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
        table = sums(ktr, r)
        s, n = look(table, ktr)
        st, nt = look(table, kte)
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_fuzzy(k, r):
        table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in (-1, 0, 1):
            a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
            s, n = s + a, n + b
            a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
            st, nt = st + a, nt + b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_decay(k, r, tau=3.0, max_gap=8):
        base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
        base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
        table = sums(base_tr + YR_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in range(-max_gap, max_gap + 1):
            w = float(np.exp(-abs(d) / tau))
            a, b = look(table, base_tr + (YR_TR + d).astype(str))
            s, n = s + w * a, n + w * b
            a, b = look(table, base_te + (YR_TE + d).astype(str))
            st, nt = st + w * a, nt + w * b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


    def f_meet(r, lam=20.0):
        table = sums(MEET_TR, r)
        s, n = look(table, MEET_TR)
        st, nt = look(table, MEET_TE)
        return (s - r) / (n - 1 + lam), st / (nt + lam)


    def stage2_features(r, kind):
        builders = {'exact': f_exact, 'fuzzy': f_fuzzy, 'decay': f_decay}
        cols_tr, cols_te, names = [], [], []
        for kd in (['exact', 'fuzzy', 'decay'] if kind == 'all' else [kind]):
            for k in PREFIX:
                a, b = builders[kd](k, r)
                cols_tr.append(a * VALID_TR)
                cols_te.append(b * VALID_TE)
                names.append(f'{kd}_{k}')
        a, b = f_meet(r)
        cols_tr.append(a)
        cols_te.append(b)
        names.append('meet')
        return np.column_stack(cols_tr), np.column_stack(cols_te), names


    alphas = {lv: fit_alpha(LEVEL == lv, base_oof, MATE) for lv in ['exact', 'core']}
    p1 = apply_twins(base_oof, MATE, LEVEL, alphas)
    p1_cf = base_oof.copy()
    for h in (0, 1):
        a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), base_oof, MATE) for lv in ['exact', 'core']}
        m = HALF == h
        p1_cf[m] = apply_twins(base_oof[m], MATE[m], LEVEL[m], a_h)
    t1 = apply_twins(base_test, T_MATE, T_LEVEL, alphas)
    r1 = y - p1
    FEATS = {kind: stage2_features(r1, kind) for kind in ['exact', 'fuzzy', 'decay', 'all']}
    GBM_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=300, subsample=0.8,
                      subsample_freq=1, reg_lambda=10.0, random_state=SEED, n_jobs=-1, verbose=-1)


    def linear_fit_predict(F_fit, r_fit, F_apply):
        beta = np.linalg.lstsq(F_fit, r_fit, rcond=None)[0]
        return F_apply @ beta


    def gbm_fit_predict(F_fit, r_fit, F_apply):
        return lgb.LGBMRegressor(**GBM_PARAMS).fit(F_fit, r_fit).predict(F_apply)


    S2_VARIANTS = {'S0_exact': ('exact', linear_fit_predict), 'S1_fuzzy': ('fuzzy', linear_fit_predict),
                   'S2_decay': ('decay', linear_fit_predict), 'S3_all_linear': ('all', linear_fit_predict),
                   'S4_all_gbm': ('all', gbm_fit_predict)}
    rows, S2_OOF, S2_TEST = [], {}, {}
    for name, (kind, fn) in S2_VARIANTS.items():
        F, FT, names = FEATS[kind]
        pred_cf = p1_cf.copy()
        for h in (0, 1):
            fit, m = HALF != h, HALF == h
            pred_cf[m] = p1_cf[m] + fn(F[fit], r1[fit], F[m])
        S2_OOF[name], S2_TEST[name] = pred_cf, t1 + fn(F, r1, FT)
        rows.append({'variant': name, 'n_features': len(names), 'crossfit_rmse': rmse(y, pred_cf),
                     'gain_vs_twins': rmse(y, p1_cf) - rmse(y, pred_cf),
                     **{f'fold{k}': v for k, v in fold_rmse(pred_cf).items()}})
    s2_table = pd.DataFrame(rows).set_index('variant').sort_values('crossfit_rmse')
    print(f'{FINAL} 融合：OOF {rmse(y, base_oof):.3f} → 加双胞胎（交叉拟合）{rmse(y, p1_cf):.3f}')
    display(s2_table.round(3))
    s2_table.to_csv(OUTPUT_DIR / 'summary_stage2_variants.csv')
    BEST_S2 = s2_table.index[0]
    for name in dict.fromkeys(['S0_exact', BEST_S2]):
        path = OUTPUT_DIR / f'submission_{FINAL}_blend3_twins_{name}.csv'
        pd.DataFrame({'id': test['id'], TARGET: np.clip(S2_TEST[name], 0, None)}).to_csv(path, index=False)
        print('写出', path.name)
    print(f'交叉拟合最好的第二层：{BEST_S2}')


    def bucket(values, edges, labels):
        return pd.cut(values, edges, labels=labels, right=False).astype(str)


    groups = {
        'age_half': P['age_half'].iloc[:n_train].to_numpy(),
        'ath_n': bucket(H['ath_n'].iloc[:n_train].fillna(-1), [-1, 0, 2, 3, 6, 1e9], ['未知', '1', '2', '3-5', '6+']),
        'ath_years_since_first': bucket(H['ath_years_since_first'].iloc[:n_train].fillna(-1), [-1, 0, 1, 2, 5, 100],
                                        ['未知', '0', '1', '2-4', '5+']),
        'ath_ever_abroad': H2['ath_ever_abroad'].iloc[:n_train].fillna(-1).to_numpy(),
        'ath_max_meet_states': bucket(H2['ath_max_meet_states'].iloc[:n_train].fillna(-1), [-1, 0, 1, 2, 5, 10, 1e9],
                                      ['未知', '0', '1', '2-4', '5-9', '10+']),
        'meet_n_states': bucket(M2['meet_n_states'].iloc[:n_train], [0, 1, 2, 5, 10, 1e9], ['0', '1', '2-4', '5-9', '10+']),
        'dup_exact_n>0': (D['dup_exact_n'].iloc[:n_train] > 0).to_numpy(),
        'Sex×Equipment': (train['Sex'] + '|' + train['Equipment']).to_numpy(),
        'Tested': train['Tested'].to_numpy(),
        'MeetYear 段': bucket(train['MeetYear'], [1900, 2000, 2010, 2015, 2020, 2030],
                              ['<2000', '2000s', '2010-14', '2015-19', '2020+']),
    }
    columns = {'R2': BLENDS['R2']['oof'], FINAL: BLENDS[FINAL]['oof'], f'{FINAL}+双胞胎+{BEST_S2}': S2_OOF[BEST_S2]}
    frames = []
    for label, pred in columns.items():
        res = y - pred
        parts = []
        for g, key in groups.items():
            part = pd.DataFrame({'k': key, 'r': res}).groupby('k')['r'].agg(['size', 'mean'])
            part.index = pd.MultiIndex.from_product([[g], part.index.astype(str)], names=['signal', 'level'])
            parts.append(part)
        frames.append(pd.concat(parts).rename(columns={'mean': label}))
    diag = pd.concat(frames, axis=1)
    diag = diag.loc[:, ~diag.columns.duplicated()]
    display(diag.round(2))
    diag.to_csv(OUTPUT_DIR / 'diagnostics_residuals.csv')
    print('完成。')
else:
    print("训练开关关闭（TRAIN['round3'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_round3():
    directory = ARTIFACTS / '02_feature_engineering' / 'round3'
    singles = {m: read_summary(directory / f'summary_single_{m}.csv') for m in ['lgb', 'xgb', 'catboost']}
    singles = {m: t for m, t in singles.items() if t is not None}
    if singles:
        print('单模型：相对 R2 的平均 ΔRMSE（kg；PM 不改变 LightGBM 的特征）：')
        display(pd.DataFrame({m: t['mean_delta'] for m, t in singles.items()}).reindex(['H2', 'E', 'PM']).round(3))
    screen = read_summary(directory / 'summary_blend_screen.csv')
    print('三模型 NNLS 融合：')
    show(screen, ['oof_rmse', 'mean_delta', 'folds_better', 'keep', 'weights_lgb_xgb_cb'])
    plot_fold_deltas(screen, 'R2', 'Round 3: three-model blend vs R2')
    print('三个块都达标，合成 ALL3：')
    show(read_summary(directory / 'summary_blend_all3.csv'), ['oof_rmse', 'mean_delta', 'folds_better', 'keep'])


show_round3()

**结论**：
- `H2` −0.09（5/5 折）、`E` −0.07（4/5）、`PM` −0.11（5/5），**ALL3 合计 −0.20（5/5 折）**，三模型融合 76.876。
- 残差诊断显示剩下的分组偏差多在 2 kg 以内，不看成绩的特征到此基本饱和。

### 5.4 第四轮（实验 14，基线 R3 = 实验 11 的最终分模型特征）

三个模型改为 LightGBM、XGBoost 和最强的 cb_rel（DOTS 相对力量目标的 CatBoost，见 03）。新特征和 04 的第二层平滑有重叠，所以还要看加平滑之后的净收益。

同一个训练开关还包括 `id` 结构检查（结论见 01）和目标变量工程（结果在 03）。

In [ ]:
if TRAIN['round4']:
    ensure_training_env()
    # 实验 14 的设置（结果目录：原 artifacts/feature_round4_v1）；基线 R3 = 实验 11 的最终分模型特征
    OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / 'round4'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    SEEDS = [42, 7, 2024]                      # 每个种子：不同的五折划分、内层早停划分和模型随机性
    FINAL_MODELS = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm']   # 每折内的运行顺序：快的先跑
    FULL_REFIT = True                          # 树模型在全部训练行上重训
    REFIT_FACTOR = 1.2                         # 全量重训轮数 = CV 平均最佳轮数 × 1.2（数据多约 39%）

    SCREEN_MODELS = ['lgb', 'xgb', 'cb_rel']        # 筛选用的三个模型（cb_rel 是最强的单模型）
    TARGET_MODELS = ['cb_allo', 'cb_gl']            # 目标变量工程：两种力量公式
    INNER_FEATURE_FOLDS = 5                         # 基于成绩的特征：外层训练部分内的嵌套折数
    # ---- 最终特征：10 的 ALL3（三个块都达标）----
    _FRAMES3.clear()
    VARIANTS3['ALL3'] = ['H2', 'E', 'PM']
    FAMILY = {'lgb': 'lgb', 'lgb_rel': 'lgb', 'tabm': 'lgb', 'xgb': 'xgb', 'catboost': 'catboost', 'cb_rel': 'catboost'}
    FRAMES = {m: frames3('ALL3', FAMILY[m]) for m in FINAL_MODELS}
    for m, (Xtr_m, Xte_m, cats_m) in FRAMES.items():
        print(f'{m:9s} {Xtr_m.shape[1]} 列（{len(cats_m)} 个类别列），块：{blocks_for("ALL3", FAMILY[m])}')


    from scipy.stats import spearmanr

    ids_all = pd.concat([train['id'], test['id']], ignore_index=True).to_numpy()
    order = np.argsort(ids_all)
    perm = np.random.default_rng(0).permutation(len(ids_all))
    checks = {'同一场比赛': meet.astype(str).to_numpy(), '同一联盟': both['Federation'].astype(str).to_numpy(),
              '同一比赛年份': both['MeetYear'].to_numpy(), '同一国家': both['Country'].astype(str).to_numpy()}
    rows = []
    for name, values in checks.items():
        by_id = float((values[order][1:] == values[order][:-1]).mean())
        random = float((values[perm][1:] == values[perm][:-1]).mean())
        rows.append({'相邻两行': name, '按 id 排序': by_id, '随机顺序': random, '倍数': by_id / max(random, 1e-12)})
    id_table = pd.DataFrame(rows).set_index('相邻两行')
    display(id_table.round(4))
    rho = float(spearmanr(ids_all, both['MeetYear'].to_numpy())[0])
    print(f'id 与比赛年份的 Spearman 相关：{rho:.4f}')
    ID_STRUCTURE = bool(id_table.loc['同一场比赛', '倍数'] >= 1.5 or abs(rho) >= 0.1)
    print('id 含有比赛结构：', ID_STRUCTURE, '→', '加入 ID 变体' if ID_STRUCTURE else '不测试 id（顺序是随机的）')
    id_table.to_csv(OUTPUT_DIR / 'summary_id_check.csv')


    N_ALL = len(both)
    TEST_ROWS = np.arange(n_train, N_ALL)
    REL_ALL = np.full(N_ALL, np.nan)
    REL_ALL[:n_train] = y / dots_scale(train['Sex'], train['BodyweightKg'])    # 相对力量（DOTS 分）
    TOTAL_ALL = np.full(N_ALL, np.nan)
    TOTAL_ALL[:n_train] = y
    BIRTH_ALL = pd.Series(np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).to_numpy())
    YEAR_ALL = pd.Series(both['MeetYear'].astype(int).to_numpy())
    BW_ALL = both['BodyweightKg'].to_numpy(dtype='float64')
    VALID_ALL = (both['Country'] != 'Unknown').to_numpy()


    def text_key_all(columns):
        out = both[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + both[c].astype(str)
        return out.reset_index(drop=True)


    PFX_ALL = {'A0': text_key_all(['Sex', 'Country', 'State']),
               'A1': text_key_all(['Sex', 'Country', 'State', 'Federation']),
               'A2': text_key_all(['Sex', 'Country', 'State', 'Federation', 'Equipment'])}
    KB_SHIFT = {k: {d: p + '|' + (BIRTH_ALL + d).astype(str) for d in (-1, 0, 1)} for k, p in PFX_ALL.items()}
    KB_A1 = KB_SHIFT['A1'][0]
    KBY_A1 = KB_A1 + '|' + YEAR_ALL.astype(str)
    DECAY_GAPS = range(-8, 9)
    KBY_SHIFT = {d: KB_A1 + '|' + (YEAR_ALL + d).astype(str) for d in DECAY_GAPS}
    MEET_ALL = {'meet': meet.reset_index(drop=True).astype(str),
                'div': text_key_all(['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment'])}
    EXACT_ALL = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy()).astype(str)
    CORE_ALL = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy()).astype(str)


    def agg(keys, values, ref, funcs=('sum', 'count', 'max')):
        frame = pd.DataFrame({'k': keys.iloc[ref].to_numpy(), 'v': values[ref]})
        return frame.groupby('k')['v'].agg(list(funcs))


    def lookup(table, column, keys, fill):
        return keys.map(table[column]).fillna(fill).to_numpy(dtype='float64')


    def athlete_block(ref, query):
        """ref：有成绩的参考行（训练行号）；query：要算特征的行（both 中的行号，不在 ref 里）。"""
        cols, m = {}, len(query)
        for k in ['A0', 'A1', 'A2']:
            table = agg(KB_SHIFT[k][0], REL_ALL, ref)
            bw_table = agg(KB_SHIFT[k][0], BW_ALL, ref, ('sum',))
            s, n, bws, mx = np.zeros(m), np.zeros(m), np.zeros(m), np.full(m, -np.inf)
            for d in (-1, 0, 1):                                       # 出生年 ±1
                kq = KB_SHIFT[k][d].iloc[query]
                s, n = s + lookup(table, 'sum', kq, 0), n + lookup(table, 'count', kq, 0)
                mx = np.maximum(mx, lookup(table, 'max', kq, -np.inf))
                bws = bws + lookup(bw_table, 'sum', kq, 0)
            has = n > 0
            cols[f'as_{k}_n'] = n
            cols[f'as_{k}_mean'] = np.where(has, s / np.maximum(n, 1), np.nan)
            cols[f'as_{k}_max'] = np.where(has, mx, np.nan)
            if k == 'A1':
                cols['as_A1_bw_diff'] = np.where(has, BW_ALL[query] - bws / np.maximum(n, 1), np.nan)
        table = agg(KBY_A1, REL_ALL, ref, ('sum', 'count'))          # 时间衰减与年份差（A1，精确出生年）
        ws, wn, gap_sum, gap_n, gap_min = np.zeros(m), np.zeros(m), np.zeros(m), np.zeros(m), np.full(m, np.inf)
        for d in DECAY_GAPS:
            kq = KBY_SHIFT[d].iloc[query]
            sd, nd = lookup(table, 'sum', kq, 0), lookup(table, 'count', kq, 0)
            w = float(np.exp(-abs(d) / 3.0))
            ws, wn = ws + w * sd, wn + w * nd
            gap_sum, gap_n = gap_sum + abs(d) * nd, gap_n + nd
            gap_min = np.where(nd > 0, np.minimum(gap_min, abs(d)), gap_min)
        cols['as_A1_decay'] = np.where(wn > 0, ws / np.maximum(wn, 1e-12), np.nan)
        cols['as_A1_gap_mean'] = np.where(gap_n > 0, gap_sum / np.maximum(gap_n, 1), np.nan)
        cols['as_A1_gap_min'] = np.where(np.isfinite(gap_min), gap_min, np.nan)
        cols['as_same_equipment_share'] = np.where(cols['as_A1_n'] > 0,
                                                   cols['as_A2_n'] / np.maximum(cols['as_A1_n'], 1), np.nan)
        frame = pd.DataFrame(cols)
        frame.loc[~VALID_ALL[query]] = np.nan                           # 国家未知时不做运动员匹配
        for name, keys in [('tw_exact', EXACT_ALL), ('tw_core', CORE_ALL)]:
            t = agg(keys, TOTAL_ALL, ref, ('sum', 'count'))
            kq = keys.iloc[query]
            s, n = lookup(t, 'sum', kq, 0), lookup(t, 'count', kq, 0)
            frame[f'{name}_total'] = np.where(n > 0, s / np.maximum(n, 1), np.nan)
            frame[f'{name}_n'] = n
        return frame


    def meet_block(ref, query):
        cols = {}
        for name, keys in MEET_ALL.items():
            t = agg(keys, REL_ALL, ref)
            kq = keys.iloc[query]
            s, n, mx = lookup(t, 'sum', kq, 0), lookup(t, 'count', kq, 0), lookup(t, 'max', kq, np.nan)
            cols[f'ms_{name}_n'] = n
            cols[f'ms_{name}_mean'] = np.where(n > 0, s / np.maximum(n, 1), np.nan)
            cols[f'ms_{name}_max'] = np.where(n > 0, mx, np.nan)
        return pd.DataFrame(cols)


    BLOCK_FUNCS = {'AS': athlete_block, 'MS': meet_block}
    _FOLD_FEATS = {}


    def nested_features(block, fold):
        """(训练行特征 n_train 行, 测试行特征)：外层训练部分用内层折外，外层验证行和测试行用整个训练部分。"""
        key = (block, fold)
        if key not in _FOLD_FEATS:
            fn = BLOCK_FUNCS[block]
            outer_train, outer_valid = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
            parts = []
            inner = KFold(INNER_FEATURE_FOLDS, shuffle=True, random_state=100 + fold)
            for inner_ref, inner_query in inner.split(outer_train):
                part = fn(outer_train[inner_ref], outer_train[inner_query])
                part.index = outer_train[inner_query]
                parts.append(part)
            part = fn(outer_train, outer_valid)
            part.index = outer_valid
            parts.append(part)
            train_part = pd.concat(parts).sort_index()
            require(train_part.index.equals(pd.RangeIndex(n_train)), f'{block} 第 {fold} 折的嵌套特征行数不符')
            test_part = fn(outer_train, TEST_ROWS).reset_index(drop=True)
            _FOLD_FEATS[key] = (train_part, test_part)
        return _FOLD_FEATS[key]


    ID_TRAIN = pd.DataFrame({'row_id': train['id'].to_numpy(dtype='float64')})
    ID_TEST = pd.DataFrame({'row_id': test['id'].to_numpy(dtype='float64')})
    VARIANTS4 = {'R3': [], 'AS': ['AS'], 'MS': ['MS']}
    if ID_STRUCTURE:
        VARIANTS4['ID'] = ['ID']
    FAMILY_OF = {'lgb': 'lgb', 'xgb': 'xgb', 'cb_rel': 'cb_rel', 'cb_allo': 'cb_rel', 'cb_gl': 'cb_rel'}


    def frames_aug(variant, model, fold):
        base_tr, base_te, _ = FRAMES[FAMILY_OF[model]]
        blocks = VARIANTS4[variant]
        extra_tr = [nested_features(b, fold)[0] for b in blocks if b in BLOCK_FUNCS]
        extra_te = [nested_features(b, fold)[1] for b in blocks if b in BLOCK_FUNCS]
        if 'ID' in blocks:
            extra_tr.append(ID_TRAIN)
            extra_te.append(ID_TEST)
        if not extra_tr:
            return base_tr, base_te
        return pd.concat([base_tr] + extra_tr, axis=1), pd.concat([base_te] + extra_te, axis=1)


    started = time.perf_counter()
    demo_tr, demo_te = nested_features('AS', 1)
    print(f'AS 第 1 折：{demo_tr.shape[1]} 列，用时 {time.perf_counter() - started:.0f}s')
    display(pd.DataFrame({'train_nonnull': demo_tr.notna().mean(), 'test_nonnull': demo_te.notna().mean(),
                          'train_mean': demo_tr.mean(), 'test_mean': demo_te.mean()}).round(3))
    corr = pd.Series({c: np.corrcoef(demo_tr[c].fillna(demo_tr[c].median()), y)[0, 1] for c in demo_tr.columns})
    print('与 TotalKg 的相关（第 1 折，训练行为嵌套折外值）：'); display(corr.round(3).sort_values())


    LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, cat_smooth=20, verbose=-1)
    XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                      reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                      max_cat_to_onehot=1, eval_metric='rmse')
    CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                     border_count=128)
    TABM_PARAMS = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0, num_emb_type='pwl')
    TABM_FALLBACK = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0)
    PATIENCE = {'lgb': 200, 'lgb_rel': 200, 'xgb': 200, 'catboost': 300, 'cb_rel': 300}
    GBDT = [m for m in FINAL_MODELS if m != 'tabm']

    SCALE = dots_scale(train['Sex'], train['BodyweightKg'])           # dots_scale 来自 09 的特征单元
    SCALE_TEST = dots_scale(test['Sex'], test['BodyweightKg'])
    require(min(SCALE.min(), SCALE_TEST.min()) > 0.3, 'DOTS 系数异常')

    X_NN, XT_NN, CATS_NN = FRAMES['tabm']
    NUM_NN = [c for c in X_NN.columns if c not in CATS_NN]
    NAN_NN = [c for c in NUM_NN if X_NN[c].isna().any() or XT_NN[c].isna().any()]


    def target_and_weight(model, rows):
        if model.endswith('_rel'):
            return y[rows] / SCALE[rows], SCALE[rows] ** 2       # 相对力量误差 × scale = kg 误差
        return y[rows], None


    def gbdt_fit(model, seed, Xa, ya, wa, Xs=None, ys=None, ws=None, rounds=MAX_ROUNDS):
        """Xs 为 None 时不早停（全量重训）。返回 (模型, 最佳轮数)。"""
        cats = FRAMES[model][2]
        if model in ('lgb', 'lgb_rel'):
            m = lgb.LGBMRegressor(n_estimators=rounds, random_state=seed, n_jobs=-1, **LGB_PARAMS)
            if Xs is None:
                m.fit(Xa, ya, sample_weight=wa)
                return m, rounds
            m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xs, ys)], eval_sample_weight=None if ws is None else [ws],
                  callbacks=[lgb.early_stopping(PATIENCE[model], verbose=False)])
            return m, int(m.best_iteration_ or rounds)
        if model == 'xgb':
            if Xs is None:
                m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, **XGB_PARAMS)
                m.fit(Xa, ya, verbose=False)
                return m, rounds
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, early_stopping_rounds=PATIENCE['xgb'],
                                 **XGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
            return m, int(m.best_iteration) + 1
        common = dict(iterations=rounds, loss_function='RMSE', eval_metric='RMSE', random_seed=seed, verbose=0,
                      task_type='GPU', devices='0', allow_writing_files=False, **CB_PARAMS)
        if Xs is None:
            m = catboost.CatBoostRegressor(**common)
            m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa))
            return m, rounds
        m = catboost.CatBoostRegressor(od_type='Iter', od_wait=PATIENCE[model], **common)
        m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa),
              eval_set=catboost.Pool(Xs, ys, cat_features=cats, weight=ws), use_best_model=True)
        return m, int(m.get_best_iteration()) + 1


    def gbdt_predict(model, m, best, X):
        if model in ('lgb', 'lgb_rel'):
            return m.predict(X, num_iteration=best)
        if model == 'xgb':
            return m.predict(X, iteration_range=(0, best))
        return m.predict(X)


    def nn_frames(*frames):
        """数值缺失用第一个表（外层训练部分）的中位数填补并加缺失标记；类别保持 category。"""
        median = frames[0][NUM_NN].median()
        out = []
        for frame in frames:
            z = frame.copy()
            for c in NAN_NN:
                z[f'{c}_isna'] = z[c].isna().astype('float64')
            z[NUM_NN] = z[NUM_NN].fillna(median).fillna(0.0)
            out.append(z.reset_index(drop=True))
        return out


    def tabm_fit_predict(seed, a, s, v, Xt, overrides=None):
        require(TabM_D_Regressor is not None, f'pytabkit 不可用：{PYTABKIT_ERROR}')
        params = dict(TABM_PARAMS, random_state=seed, **(overrides or {}))
        unknown = set(params) - set(inspect.signature(TabM_D_Regressor.__init__).parameters)
        require(not unknown, f'TabM 不支持参数 {sorted(unknown)}')
        Za, Zs, Zv, Zt = nn_frames(X_NN.iloc[a], X_NN.iloc[s], X_NN.iloc[v], Xt)
        mu, sd = y[a].mean(), y[a].std()
        m = TabM_D_Regressor(**params)
        m.fit(Za, (y[a] - mu) / sd, Zs, (y[s] - mu) / sd, cat_col_names=[c for c in Za.columns if c in CATS_NN])
        p_val = np.asarray(m.predict(Zv), dtype='float64').reshape(-1) * sd + mu
        p_test = np.asarray(m.predict(Zt), dtype='float64').reshape(-1) * sd + mu
        del m
        gc.collect()
        torch.cuda.empty_cache()
        return p_val, p_test


    def fit_fold(model, seed, a, s, v, test_rows=None, rounds=MAX_ROUNDS, overrides=None):
        """a / s / v：内层拟合 / 内层早停 / 外层验证 的训练行号。返回 (验证预测, 测试预测, 最佳轮数或 None)。"""
        Xtr, Xte, _ = FRAMES[model]
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
        if model == 'tabm':
            p_val, p_test = tabm_fit_predict(seed, a, s, v, Xt, overrides)
            return p_val, p_test, None
        ya, wa = target_and_weight(model, a)
        ys, ws = target_and_weight(model, s)
        if wa is not None:
            norm = wa.mean()
            wa, ws = wa / norm, ws / norm
        m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
        p_val, p_test = gbdt_predict(model, m, best, Xtr.iloc[v]), gbdt_predict(model, m, best, Xt)
        if model.endswith('_rel'):
            p_val, p_test = np.asarray(p_val) * SCALE[v], np.asarray(p_test) * scale_t
        return np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64'), best


    def fit_full(model, seed, rounds, test_rows=None):
        """全部训练行、固定轮数、不早停。返回测试预测。"""
        Xtr, Xte, _ = FRAMES[model]
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
        rows = np.arange(n_train)
        ya, wa = target_and_weight(model, rows)
        if wa is not None:
            wa = wa / wa.mean()
        m, _ = gbdt_fit(model, seed, Xtr, ya, wa, rounds=rounds)
        p_test = gbdt_predict(model, m, rounds, Xt)
        if model.endswith('_rel'):
            p_test = np.asarray(p_test) * scale_t
        return np.asarray(p_test, dtype='float64')


    def gl_scale(sex_values, equipment_values, bw_values):
        sex = np.asarray(sex_values).astype(str)
        equipped = ~np.isin(np.asarray(equipment_values).astype(str), ['Raw', 'Wraps'])
        bw = np.maximum(np.asarray(bw_values, dtype='float64'), 30.0)
        coef = {('M', False): (1199.72839, 1025.18162, 0.00921), ('M', True): (1236.25115, 1449.21864, 0.01644),
                ('F', False): (610.32796, 1045.59282, 0.03048), ('F', True): (758.63878, 949.31382, 0.02435)}

        def curve(s, e):
            a, b, c = coef[(s, e)]
            return a - b * np.exp(-c * bw)

        male = np.where(equipped, curve('M', True), curve('M', False))
        female = np.where(equipped, curve('F', True), curve('F', False))
        return np.where(sex == 'M', male, np.where(sex == 'F', female, 0.5 * (male + female))) / 100.0


    ALLO = (train['BodyweightKg'].to_numpy(dtype='float64') / 80.0) ** (2 / 3)
    ALLO_TEST = (test['BodyweightKg'].to_numpy(dtype='float64') / 80.0) ** (2 / 3)
    GL = gl_scale(train['Sex'], train['Equipment'], train['BodyweightKg'])
    GL_TEST = gl_scale(test['Sex'], test['Equipment'], test['BodyweightKg'])
    require(min(GL.min(), GL_TEST.min()) > 0 and min(ALLO.min(), ALLO_TEST.min()) > 0, '力量系数异常')
    SCALE_OF = {'cb_rel': (SCALE, SCALE_TEST), 'cb_allo': (ALLO, ALLO_TEST), 'cb_gl': (GL, GL_TEST)}
    for extra in TARGET_MODELS:
        FRAMES[extra] = FRAMES['cb_rel']
        PATIENCE[extra] = PATIENCE['cb_rel']


    def fit_matrix(model, Xtr, Xte, a, s, v, seed=SEED, rounds=MAX_ROUNDS, test_rows=None):
        """用给定的特征表训练（内层拟合 a、早停 s），返回 (外层验证预测, 测试预测, 最佳轮数)。"""
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        if model in SCALE_OF:
            scale, scale_test = SCALE_OF[model]
            scale_t = scale_test if test_rows is None else scale_test[test_rows]
            ya, ys = y[a] / scale[a], y[s] / scale[s]
            norm = (scale[a] ** 2).mean()
            wa, ws = scale[a] ** 2 / norm, scale[s] ** 2 / norm
        else:
            ya, ys, wa, ws = y[a], y[s], None, None
        m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
        p_val = np.asarray(gbdt_predict(model, m, best, Xtr.iloc[v]), dtype='float64')
        p_test = np.asarray(gbdt_predict(model, m, best, Xt), dtype='float64')
        if model in SCALE_OF:
            p_val, p_test = p_val * scale[v], p_test * scale_t
        return p_val, p_test, best


    rng = np.random.default_rng(SEED)
    probe = rng.choice(np.flatnonzero(FOLD_IDS != 1), 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    failed = {}
    checks = [(m, v) for v in VARIANTS4 for m in SCREEN_MODELS] + [(m, 'R3') for m in TARGET_MODELS]
    for model, variant in checks:
        try:
            Xtr, Xte = frames_aug(variant, model, 1)
            p_val, p_test, _ = fit_matrix(model, Xtr, Xte, pa, ps, pv, rounds=30, test_rows=np.arange(300))
            require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
            print(f'预检通过：{model:8s} {variant:3s}（{Xtr.shape[1]} 列）RMSE@30轮 {rmse(y[pv], p_val):.1f}')
        except Exception as exc:
            failed[(model, variant)] = repr(exc)
            print(f'预检失败，跳过：{model} {variant} → {exc!r}')
    require(not any(v == 'R3' and m in SCREEN_MODELS for (m, v) in failed), '基线 R3 预检失败，请检查环境')


    def run_one(model, variant, fold):
        stem = f'{model}__{variant}__fold{fold}'
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        if json_path.is_file() and npz_path.is_file():
            return json.loads(json_path.read_text(encoding='utf-8')), 'cache'
        Xtr, Xte = frames_aug(variant, model, fold)
        fit_idx, valid_idx = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
        inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 07–13
        started = time.perf_counter()
        p_val, p_test, best = fit_matrix(model, Xtr, Xte, inner_fit, inner_stop, valid_idx)
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
        np.savez(npz_path, valid_idx=valid_idx, val=p_val, test=p_test)
        info = {'model': model, 'variant': variant, 'fold': fold, 'blocks': VARIANTS4[variant],
                'n_features': int(Xtr.shape[1]), 'best_iteration': best, 'rmse': rmse(y[valid_idx], p_val),
                'seconds': time.perf_counter() - started, 'gpu': GPU_NAME,
                'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        json_path.write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
        return info, f'{info["seconds"]:.0f}s'


    def run_variants(variants, models):
        for fold in range(1, N_FOLDS + 1):
            fold_started = time.perf_counter()
            for model in models:
                for variant in variants:
                    if (model, variant) in failed:
                        continue
                    info, how = run_one(model, variant, fold)
                    print(f'fold {fold}  {model:8s} {variant:4s}  RMSE {info["rmse"]:.3f}  '
                          f'最佳轮数 {info["best_iteration"]:5d}  ({how})')
            print(f'—— fold {fold} 完成，本次运行用时 {(time.perf_counter() - fold_started) / 60:.1f} 分钟')


    run_variants(list(VARIANTS4), SCREEN_MODELS)


    def load_oof(model, variant):
        oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
        for fold in range(1, N_FOLDS + 1):
            path = OUTPUT_DIR / f'{model}__{variant}__fold{fold}.npz'
            if not path.is_file():
                return None
            saved = np.load(path)
            oof[saved['valid_idx']] = saved['val']
            test_pred += saved['test'] / N_FOLDS
        return oof, test_pred


    def fold_rmse(pred):
        return pd.Series({k: rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)})


    def paired_table(preds, base='R3'):
        base_folds = fold_rmse(preds[base])
        rows = []
        for name, pred in preds.items():
            delta = fold_rmse(pred) - base_folds
            rows.append({'variant': name, 'oof_rmse': rmse(y, pred), 'mean_delta': delta.mean(),
                         'folds_better': int((delta < 0).sum()), **{f'd_fold{k}': v for k, v in delta.items()}})
        table = pd.DataFrame(rows).set_index('variant')
        table['keep'] = (table['mean_delta'] <= KEEP_DELTA) & (table['folds_better'] >= 4)
        table.loc[base, 'keep'] = False
        return table.sort_values('oof_rmse')


    def blend_variant(variant, models=SCREEN_MODELS):
        loaded = [load_oof(m, variant) for m in models]
        if any(item is None for item in loaded):
            return None
        P_oof = np.column_stack([o for o, _ in loaded])
        P_test = np.column_stack([t for _, t in loaded])
        w, _ = nnls(P_oof, y)
        return {'oof': P_oof @ w, 'test': P_test @ w, 'weights': [round(float(x), 3) for x in w]}


    for model in SCREEN_MODELS:
        preds = {v: load_oof(model, v)[0] for v in VARIANTS4 if load_oof(model, v) is not None}
        if 'R3' in preds:
            print(f'\n=== {model}')
            display(paired_table(preds).round(3))
    BLENDS = {v: blend_variant(v) for v in VARIANTS4}
    BLENDS = {v: b for v, b in BLENDS.items() if b is not None}
    require('R3' in BLENDS, 'R3 还没有全部完成')
    blend_table = paired_table({v: b['oof'] for v, b in BLENDS.items()})
    blend_table['weights_lgb_xgb_cbrel'] = [str(BLENDS[v]['weights']) for v in blend_table.index]
    print('\n=== 三模型 NNLS 融合（主要判读依据）')
    display(blend_table.round(3))
    blend_table.to_csv(OUTPUT_DIR / 'summary_blend_screen.csv')
    KEPT4 = [v for v in VARIANTS4 if v != 'R3' and v in blend_table.index and bool(blend_table.loc[v, 'keep'])]
    print('达标的新块：', KEPT4 or '无')
    if len(KEPT4) >= 2:
        VARIANTS4['ALL4'] = [b for v in KEPT4 for b in VARIANTS4[v]]
        run_variants(['ALL4'], SCREEN_MODELS)
        BLENDS['ALL4'] = blend_variant('ALL4')
        display(paired_table({v: BLENDS[v]['oof'] for v in ['R3'] + KEPT4 + ['ALL4']}).round(3))
    FINAL4 = 'ALL4' if 'ALL4' in BLENDS else (KEPT4[0] if KEPT4 else None)
    if FINAL4 is None and len(blend_table) > 1:
        FINAL4 = blend_table.index[0] if blend_table.index[0] != 'R3' else blend_table.index[1]
    print('与 R3 做净收益比较的变体：', FINAL4)


    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        denom = float(np.sum(d * d))
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
    TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


    def text_key(frame, columns):
        out = frame[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + frame[c].astype(str)
        return out


    BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
    BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
    YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
    PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
              'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
              'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
    SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
    PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
    PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
    MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
    MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
    VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
    VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


    def sums(keys, values):
        table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
        return table['sum'], table['count']


    def look(table, keys):
        s, n = table
        return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


    def f_exact(k, r):
        ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
        table = sums(ktr, r)
        s, n = look(table, ktr)
        st, nt = look(table, kte)
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_fuzzy(k, r):
        table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in (-1, 0, 1):
            a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
            s, n = s + a, n + b
            a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
            st, nt = st + a, nt + b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_decay(k, r, tau=3.0, max_gap=8):
        base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
        base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
        table = sums(base_tr + YR_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in range(-max_gap, max_gap + 1):
            w = float(np.exp(-abs(d) / tau))
            a, b = look(table, base_tr + (YR_TR + d).astype(str))
            s, n = s + w * a, n + w * b
            a, b = look(table, base_te + (YR_TE + d).astype(str))
            st, nt = st + w * a, nt + w * b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


    def f_meet(r, lam=20.0):
        table = sums(MEET_TR, r)
        s, n = look(table, MEET_TR)
        st, nt = look(table, MEET_TE)
        return (s - r) / (n - 1 + lam), st / (nt + lam)


    def s3_features(r):
        cols_tr, cols_te = [], []
        for builder in (f_exact, f_fuzzy, f_decay):
            for k in PREFIX:
                a, b = builder(k, r)
                cols_tr.append(a * VALID_TR)
                cols_te.append(b * VALID_TE)
        a, b = f_meet(r)
        cols_tr.append(a)
        cols_te.append(b)
        return np.column_stack(cols_tr), np.column_stack(cols_te)




    def stage2(oof, test_pred):
        alphas = {lv: fit_alpha(LEVEL == lv, oof, MATE) for lv in ['exact', 'core']}
        p1 = apply_twins(oof, MATE, LEVEL, alphas)
        p1_cf = oof.copy()
        for h in (0, 1):
            a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), oof, MATE) for lv in ['exact', 'core']}
            m = HALF == h
            p1_cf[m] = apply_twins(oof[m], MATE[m], LEVEL[m], a_h)
        r1 = y - p1
        F, FT = s3_features(r1)
        p2_cf = p1_cf.copy()
        for h in (0, 1):
            fit, m = HALF != h, HALF == h
            p2_cf[m] = p1_cf[m] + F[m] @ np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
        beta = np.linalg.lstsq(F, r1, rcond=None)[0]
        test_final = apply_twins(test_pred, T_MATE, T_LEVEL, alphas) + FT @ beta
        return {'oof': rmse(y, oof), '+双胞胎': rmse(y, p1_cf), '+S3': rmse(y, p2_cf),
                'fold_final': fold_rmse(p2_cf), 'test_final': test_final}


    net_rows, NET = [], {}
    for name in [v for v in ['R3', FINAL4] if v is not None]:
        NET[name] = stage2(BLENDS[name]['oof'], BLENDS[name]['test'])
        net_rows.append({'variant': name, **{k: NET[name][k] for k in ['oof', '+双胞胎', '+S3']}})
        path = OUTPUT_DIR / f'submission_{name}_blend3_stage2.csv'
        pd.DataFrame({'id': test['id'], TARGET: np.clip(NET[name]['test_final'], 0, None)}).to_csv(path, index=False)
        print('写出', path.name)
    net_table = pd.DataFrame(net_rows).set_index('variant')
    display(net_table.round(3))
    if FINAL4 is not None:
        delta = NET[FINAL4]['fold_final'] - NET['R3']['fold_final']
        print(f'{FINAL4} 相对 R3（都加平滑后）：平均 {delta.mean():+.3f}，{int((delta < 0).sum())}/5 折更好')
    net_table.to_csv(OUTPUT_DIR / 'summary_net_after_stage2.csv')


    for model in TARGET_MODELS:
        if (model, 'R3') in failed:
            continue
        for fold in range(1, N_FOLDS + 1):
            info, how = run_one(model, 'R3', fold)
            print(f'fold {fold}  {model:8s}  RMSE {info["rmse"]:.3f}  最佳轮数 {info["best_iteration"]}  ({how})')

    base_models = SCREEN_MODELS
    P_base = np.column_stack([load_oof(m, 'R3')[0] for m in base_models])


    def nested_nnls(P):
        out = np.zeros(n_train)
        for k in range(1, N_FOLDS + 1):
            fit, va = FOLD_IDS != k, FOLD_IDS == k
            w, _ = nnls(P[fit], y[fit])
            out[va] = P[va] @ w
        return out


    base_nested = nested_nnls(P_base)
    rows = [{'model': m, 'oof': rmse(y, load_oof(m, 'R3')[0])} for m in base_models]
    for model in TARGET_MODELS:
        loaded = load_oof(model, 'R3')
        if loaded is None:
            continue
        nested = nested_nnls(np.column_stack([P_base, loaded[0]]))
        delta = fold_rmse(nested) - fold_rmse(base_nested)
        rows.append({'model': model, 'oof': rmse(y, loaded[0]),
                     'corr_with_cb_rel': float(np.corrcoef(loaded[0], load_oof('cb_rel', 'R3')[0])[0, 1]),
                     'blend_delta_nested': rmse(y, nested) - rmse(y, base_nested), 'folds_better': int((delta < 0).sum())})
    target_table = pd.DataFrame(rows).set_index('model')
    print(f'R3 三模型融合（嵌套）：{rmse(y, base_nested):.3f}')
    display(target_table.round(3))
    target_table.to_csv(OUTPUT_DIR / 'summary_target_engineering.csv')
    print('完成。')
else:
    print("训练开关关闭（TRAIN['round4'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_round4():
    directory = ARTIFACTS / '02_feature_engineering' / 'round4'
    screen = read_summary(directory / 'summary_blend_screen.csv')
    print('三模型 NNLS 融合（LightGBM、XGBoost、cb_rel）：')
    show(screen, ['oof_rmse', 'mean_delta', 'folds_better', 'keep', 'weights_lgb_xgb_cbrel'])
    plot_fold_deltas(screen, 'R3', 'Round 4 (label-based features): three-model blend vs R3')
    net = read_summary(directory / 'summary_net_after_stage2.csv')
    if net is not None:
        print('再加双胞胎 + S3 平滑（交叉拟合）之后，以及公开榜：')
        display(net.assign(公开榜=pd.Series({'R3': 76.592, 'AS': 76.802})).round(3))


show_round4()

**结论**：
- 运动员实力 `AS` +0.14（0/5 折），比赛实力 `MS` +0.52（0/5 折），**都变差**。
- 加平滑之后，以及在公开榜上（76.802 对 76.592），结论都一样。
- 可能的原因有两个：训练行的特征只参考 80% 的数据，验证行和测试行参考全部训练部分，两者分布不一致；而且这类特征噪声大，容易过拟合。
- 同样的信息放在模型之后由 S3 平滑使用，效果更好（04、05）。

## 6. 最终使用的特征

最终特征 = 第三轮的 ALL3，按模型不同（实验 11 起所有最终模型都用它）：

| 模型 | 特征块 | 列数 |
|---|---|---:|
| LightGBM、lgb_rel、TabM、sklearn HGB | F0 + P + T + M + H + H2 + E | 57 |
| XGBoost | 同上 + M2 + S + D | 78 |
| CatBoost、cb_rel、cb_gl | 同上 + M2 + C（比赛 / 运动员 ID） | 73 |

全部特征都由本 notebook 第 1–3 节的代码生成。03、04 打开训练开关时，会用同样的代码重新生成。